# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import collections, json, math, random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm.auto import tqdm

from diffusers import AutoencoderKL, DDPMScheduler, DDIMScheduler, UNet2DModel
from diffusers.training_utils import EMAModel
from torchvision.models import resnet18, ResNet18_Weights
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity as LPIPS
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

# Pytorch sanity check
print("cuda" if torch.cuda.is_available() else "CPU")

# Globals

In [ ]:
# Paths
ROOT  = Path.cwd()
DATA  = ROOT / "perio_KPT"

BASE      = DATA / "0_Baseline"
BASE_IMG  = BASE / "images"  # base images
BASE_LAB  = BASE / "labels"  # YOLO-pose (cls + box + keypoints)
BASE_ROT  = BASE / "rotating_box" # same box + tooth long-axis angle
AUX_TR    = DATA / "2_Auxiliary_Segmentation" / "train"
AUX_VA    = DATA / "2_Auxiliary_Segmentation" / "val"

CACHE = ROOT / "cache"  # patches, masks, latents, checkpoints
CKPT  = CACHE / "ckpt"
VAE_META = CKPT / "vae_meta.json"
LDM_META = CKPT / "ldm_meta.json"
FIGS  = ROOT / "figs"
for d in (CACHE, CKPT, FIGS):
    d.mkdir(parents=True, exist_ok=True)

# patch geometry
PATCH_W, PATCH_H = 128, 256  # because teeth are about 1:3 + the bone
PATCH_ASPECT = PATCH_W / PATCH_H
MARGIN = 0.15

# latent space
LATENT_F, LATENT_CH = 4, 4
LATENT_H, LATENT_W  = PATCH_H // LATENT_F, PATCH_W // LATENT_F

# annotations
BOX_CLASSES = ("Single Root", "Double Root", "Triple Root", "ARR", "PLS")
TOOTH_CLASSES = (0, 1, 2)

# keypoints
N_KP = 11
KP = dict(CEJ_M=0, BL_M=1, RL_M=2, CEJ_D=3, BL_D=4, RL_D=5, RL_C=6, FA=7, FBL_M=8, FBL_D=9, ARR=10)
KP_FLIP_IDX = (3, 4, 5, 0, 1, 2, 6, 7, 9, 8, 10)

# severity
STAGE_EDGES = (0.15, 0.33, 0.66)
STAGE_NAMES = ("Healthy", "Mild", "Moderate", "Severe")
ADVANCED_MIN = 0.33  # Moderate + Severe
SEVERE_MIN = 0.66  # definition of severe in the reference paper
PBL_MAX = 1.0  # if above this anatomically impossible

# masks
N_MASK_CLASSES = 5
MASK_NAMES = ("background", "crown", "exposed root", "embedded root", "alveolar bone")

# colors
KP_COLORS = {
    "CEJ_M": (0, 190, 255), "CEJ_D": (0, 110, 255),
    "BL_M":  (255,  60, 60), "BL_D": (255, 150,  0),
    "RL_M":  (60, 230,  60), "RL_D": (0, 165,  90), "RL_C": (160, 255, 60),
    "FA":    (255,   0, 220),
    "FBL_M": (255, 255,  0), "FBL_D": (205, 205, 0),
    "ARR":   (255, 255, 255),
}
KP_NAME = {i: n for n, i in KP.items()}

MASK_COLORS = (
    (  0,   0,   0),   # 0 background
    ( 90, 200, 255),   # 1 crown
    (255,  70,  70),   # 2 exposed root
    ( 80, 220, 120),   # 3 embedded root
    (255, 200,  60),   # 4 alveolar bone
)

# reproducibility and runtime
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16

# Utils

Support functions for reading images and parsing annotations. Each radiograph has a YOLO-pose label with class, box and 11 keypoints, plus a separate rotating-box file with the tooth long axis angle, which are parsed into pixel coordinates.

In [ ]:
# reproducibility
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

STEMS = sorted(p.stem for p in BASE_IMG.glob("*.png"))

def read_img(path):
    """Grayscale read, so RGB and grayscale sources behave identically."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

def read_pose_label(path):
    """YOLO-pose -> [{cls, box, kps}], while coordinates stay normalised."""
    objs = []
    for line in Path(path).read_text().splitlines():
        f = line.split()
        if not f:
            continue
        v = [float(x) for x in f[1:]]
        assert len(v) == 4 + 3 * N_KP, "Unexpected number of fields in YOLO pose."
        objs.append(dict(
            cls = int(float(f[0])),
            box = np.array(v[:4]),
            kps = np.array(v[4:]).reshape(N_KP, 3)
        ))
    return objs

def read_rot_angles(path):
    """Long-axis angle per box, or None where the dataset contains the literal "None"."""
    out = []
    for line in Path(path).read_text().splitlines():
        f = line.split()
        if f:
            out.append(None if f[-1] == "None" else float(f[-1]))
    return out

def read_seg_label(path):
    """Normalised tooth polygons, single class."""
    polys = []
    for line in Path(path).read_text().splitlines():
        f = line.split()
        if len(f) < 7:
            continue
        v = np.array([float(x) for x in f[1:]])
        polys.append(v[:len(v) // 2 * 2].reshape(-1, 2))
    return polys

def denorm_kps(kps, W, H):
    out = kps.copy()
    out[:, 0] *= W
    out[:, 1] *= H
    return out

def denorm_box(box, W, H):
    cx, cy, w, h = box
    return np.array([cx * W, cy * H, w * W, h * H])

Periodontitis Bone Loss (PBL) geometry follows Banks et al., 2015. for the most part. Per side, the CEJ->bone-level distance over the CEJ->apex distance, both projected on the tooth long axis. A tooth's stage is considered to be that of its worse side.

In [ ]:
def axis_unit(theta_deg):
    """Unit vector along the tooth long axis, in image pixel coords."""
    t = math.radians(theta_deg)
    return np.array([-math.sin(t), math.cos(t)])

def crown_apex(kps_px):
    """(CEJ midpoint, apex), or (None, None) if either is unavailable."""
    ap = [j for j in (KP["RL_C"], KP["RL_M"], KP["RL_D"]) if kps_px[j, 2] > 0]
    if kps_px[KP["CEJ_M"], 2] == 0 or kps_px[KP["CEJ_D"], 2] == 0 or not ap:
        return None, None
    return (kps_px[KP["CEJ_M"], :2] + kps_px[KP["CEJ_D"], :2]) / 2, kps_px[ap, :2].mean(0)

def proj(kps_px, a, b, axis):
    """|(kp_b - kp_a) . axis|, or None if either keypoint is missing."""
    if kps_px[a, 2] == 0 or kps_px[b, 2] == 0:
        return None
    return abs(float((kps_px[b, :2] - kps_px[a, :2]) @ axis))

def tooth_pbl(kps_px, theta_deg):
    """Mesial and distal PBL, each measured along the root axis."""
    axis = axis_unit(theta_deg)
    roots = [j for j in (KP["RL_M"], KP["RL_D"], KP["RL_C"]) if kps_px[j, 2] > 0]
    out = []
    for cej, bl in ((KP["CEJ_M"], KP["BL_M"]), (KP["CEJ_D"], KP["BL_D"])):
        num = proj(kps_px, cej, bl, axis)
        den = [proj(kps_px, cej, j, axis) for j in roots]
        den = [d for d in den if d is not None and d >= 1e-6]
        out.append(None if num is None or not den else num / min(den))
    return tuple(out)

def stage_of(pbl):
    """0 Healthy, 1 Mild, 2 Moderate, 3 Severe."""
    return int(np.searchsorted(STAGE_EDGES, pbl, side="right"))

Scan: one record per tooth instance, with box, long-axis angle, keypoints, both side PBLs, worst side stage.

In [ ]:
def scan_teeth():
    """One record per tooth, with both side PBLs and the worst-side stage."""
    teeth, dropped = [], []
    for stem in tqdm(STEMS, desc="scan"):
        H, W = read_img(BASE_IMG / f"{stem}.png").shape
        objs = read_pose_label(BASE_LAB / f"{stem}.txt")
        angles = read_rot_angles(BASE_ROT / f"{stem}.txt")
        assert len(objs) == len(angles), "Labels do not match angles!"

        for i, (o, ang) in enumerate(zip(objs, angles)):
            if o["cls"] not in TOOTH_CLASSES or ang is None:
                continue
            kps = denorm_kps(o["kps"], W, H)
            sides = tooth_pbl(kps, ang)
            for side, p in zip("md", sides):
                if p is not None and p > PBL_MAX:
                    dropped.append((stem, side, p))
            valid = [p for p in sides if p is not None and p <= PBL_MAX]
            if not valid:
                continue
            pbl = max(valid)
            teeth.append(dict(stem=stem, idx=i, cls=o["cls"], WH=(W, H), angle=ang,
                              box=denorm_box(o["box"], W, H), kps=kps,
                              pbl_m=sides[0], pbl_d=sides[1], pbl=pbl,
                              stage=stage_of(pbl), advanced=pbl >= ADVANCED_MIN))
    return teeth, dropped

Patch extraction: rotate the long axis to vertical, canonicalize crown up from the CEJ and apex keypoints, crop the box, resize.

In [ ]:
def extract_patch(img, box_px, theta_deg, kps_px):
    """
    Rotate so the tooth is vertical, crop to aspect 1:2, canonicalize crown-up,
    resize to PATCH_W x PATCH_H. Returns (patch, kps_in_patch, meta).
    """
    cx, cy, w, h = box_px

    # crop rectangle in original pixels, forced to exact aspect PATCH_W:PATCH_H
    hh, ww = h * (1 + MARGIN), w * (1 + MARGIN)
    h_crop = max(hh, ww / PATCH_ASPECT)
    w_crop = h_crop * PATCH_ASPECT

    # crown-up decision: does +axis already point at the root?
    cej, apex = crown_apex(kps_px)
    u = axis_unit(theta_deg)
    flipped = bool(cej is not None and float(u @ (apex - cej)) < 0)
    alpha = theta_deg + 180.0 if flipped else theta_deg

    # rotate about the box centre and recentre it in the crop
    M = cv2.getRotationMatrix2D((float(cx), float(cy)), alpha, 1.0)
    M[0, 2] += w_crop / 2 - cx
    M[1, 2] += h_crop / 2 - cy

    # warp, then downscale separately
    crop = cv2.warpAffine(img, M, (int(round(w_crop)), int(round(h_crop))),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    patch = cv2.resize(crop, (PATCH_W, PATCH_H), interpolation=cv2.INTER_AREA)

    # carry the keypoints through the transform
    sx, sy = PATCH_W / crop.shape[1], PATCH_H / crop.shape[0]
    kp = kps_px.copy()
    kp[:, :2] = (kps_px[:, :2] @ M[:, :2].T + M[:, 2]) * (sx, sy)
    kp[kps_px[:, 2] == 0, :2] = 0

    return patch, kp, dict(M=M, alpha=alpha, flipped=flipped,
                           crop_hw=(crop.shape[0], crop.shape[1]), scale=(sx, sy))

def warp_points(pts, meta):
    """Carry (N,2) image-space points into the patch frame of a given extract_patch call."""
    return (pts @ meta["M"][:, :2].T + meta["M"][:, 2]) * meta["scale"]

def poly_box_axis(poly_px):
    """(cx,cy,w,h) and theta_deg for a polygon to get the oriented box that the aux
       dataset doesn't have."""
    rect = cv2.minAreaRect(poly_px.astype(np.float32))
    (cx, cy), (a, b), _ = rect
    pts = cv2.boxPoints(rect)
    e1, e2 = pts[1] - pts[0], pts[2] - pts[1]
    d = e1 if np.linalg.norm(e1) > np.linalg.norm(e2) else e2
    d = d / np.linalg.norm(d)
    return np.array([cx, cy, min(a, b), max(a, b)]), math.degrees(math.atan2(-d[0], d[1]))


Drawing and figures.

In [ ]:
def to_canvas(img_gray):
    """Grayscale -> RGB, so cv2 can draw colour on it."""
    return cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

def scale(canvas, frac, lo):
    """Marker and line size proportional to the image."""
    return max(lo, int(round(frac * max(canvas.shape[:2]))))

def draw_kps(canvas, kps_px, only=None):
    r, th = scale(canvas, 0.006, 3), scale(canvas, 0.002, 1)
    for i in range(N_KP):
        x, y, v = kps_px[i]
        if v == 0:
            continue
        name = KP_NAME[i]
        if only and name not in only:
            continue
        cv2.circle(canvas, (int(x), int(y)), r, KP_COLORS[name], -1)
        cv2.circle(canvas, (int(x), int(y)), r, (0, 0, 0), th)
    return canvas

def draw_box(canvas, box_px, theta_deg=None, color=(255, 255, 0)):
    cx, cy, w, h = box_px
    th = scale(canvas, 0.0025, 1)
    if theta_deg is None:
        cv2.rectangle(canvas, (int(cx - w/2), int(cy - h/2)), (int(cx + w/2), int(cy + h/2)), color, th)
    else:
        u = axis_unit(theta_deg)
        p = np.array([u[1], -u[0]])                 # perpendicular to the axis
        c = np.array([cx, cy])
        pts = np.array([c - h/2*u - w/2*p, c - h/2*u + w/2*p,
                        c + h/2*u + w/2*p, c + h/2*u - w/2*p]).astype(np.int32)
        cv2.polylines(canvas, [pts], True, color, th, cv2.LINE_AA)
    return canvas

def draw_axis(canvas, kps_px, box_px, theta_deg, color=(255, 0, 255)):
    cej, apex = crown_apex(kps_px)
    if cej is None:
        return canvas
    u = axis_unit(theta_deg)
    if u @ (apex - cej) < 0:                        # binary, immune to the ~10 deg jitter
        u = -u
    cx, cy, w, h = box_px
    c = np.array([cx, cy])
    cv2.arrowedLine(canvas, tuple((c - h/2*u).astype(int)), tuple((c + h/2*u).astype(int)),
                    color, scale(canvas, 0.0025, 1), cv2.LINE_AA, tipLength=0.06)
    return canvas

def overlay_mask(canvas, mask, colors=MASK_COLORS, alpha=0.45):
    """Blend an integer label map over an RGB canvas. Label 0 is left untouched."""
    out = canvas.copy()
    for k in range(1, len(colors)):
        sel = mask == k
        if sel.any():
            out[sel] = ((1 - alpha) * out[sel] + alpha * np.array(colors[k])).astype(np.uint8)
    return out

def draw_polys(canvas, polys_px, color=(255, 90, 90), alpha=0.40):
    overlay = canvas.copy()
    for p in polys_px:
        cv2.fillPoly(overlay, [p.astype(np.int32)], color)
    cv2.addWeighted(overlay, alpha, canvas, 1 - alpha, 0, canvas)
    for p in polys_px:
        cv2.polylines(canvas, [p.astype(np.int32)], True, color, 1, cv2.LINE_AA)
    return canvas

def show_grid(images, titles=None, ncols=4, size=3.2, cmap="gray", save=None):
    n = len(images); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*size, nrows*size*1.15))
    flat = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(flat, images, titles or [""] * n):
        ax.imshow(im, cmap=cmap if im.ndim == 2 else None)
        ax.set_title(t, fontsize=8); ax.axis("off")
    for ax in flat[n:]:
        ax.axis("off")
    fig.tight_layout()
    if save:
        fig.savefig(FIGS / save, dpi=130, bbox_inches="tight")
    return fig

def norm_img(a):
    """uint8 [0,255] -> float32 [-1,1]."""
    return a.astype(np.float32) / 127.5 - 1.0

def denorm_img(t):
    """float32 [-1,1] -> uint8 [0,255], for display and for FID/LPIPS."""
    return np.clip((np.asarray(t) + 1.0) * 127.5, 0, 255).astype(np.uint8)

Losses:
- Dice+BCE for the binary segmenter,
- Dice+CE for the 5-class segmenter,
- reconstruction+LPIPS+KL+hinge for the VAE.
The adversarial weight is set by gradient-norm matching (Esser et al., 2021).

In [ ]:
lpips_fn = LPIPS(net_type="vgg", normalize=False).to(DEVICE)
for p in lpips_fn.parameters():
    p.requires_grad_(False)

def perceptual(a, b):
    """LPIPS on [-1,1] 1-channel images. reset() or scores accumulate across steps."""
    v = lpips_fn(a.clamp(-1, 1).repeat(1, 3, 1, 1), b.clamp(-1, 1).repeat(1, 3, 1, 1))
    lpips_fn.reset()
    return v

def hinge_d(real_score, fake_score):
    """Hinge, not BCE: it stops pushing past margin 1, so the discriminator cannot run
       away from the generator and saturate its gradients."""
    return 0.5 * (F.relu(1.0 - real_score).mean() + F.relu(1.0 + fake_score).mean())

def adaptive_w(l_rec, l_adv, last_layer):
    """Esser et al. 2021: scale the adversarial term so its gradient at the output layer
       matches the reconstruction one. Turns the adversarial weight into a measurement."""
    g_rec = torch.autograd.grad(l_rec, last_layer, retain_graph=True)[0].float()
    g_adv = torch.autograd.grad(l_adv, last_layer, retain_graph=True)[0].float()
    return (g_rec.norm() / (g_adv.norm() + 1e-4)).clamp(0.0, 1e4).detach()

def vae_loss(vae, x, disc=None, w_lpips=0.1, w_kl=1e-6, w_adv=0.5):
    """L1 + LPIPS + KL, plus the patch-adversarial term once disc is passed.
       L1 alone is minimised by the conditional median, i.e. flat grey under uncertainty;
       the discriminator asks instead whether the texture is distinguishable from bone."""
    post = vae.encode(x).latent_dist
    z    = post.sample()
    rec  = vae.decode(z).sample
    l1   = (rec.float() - x).abs().mean()
    lp   = perceptual(rec.float(), x)
    kl   = post.kl().mean()
    l_rec = l1 + w_lpips * lp
    loss  = l_rec + w_kl * kl
    parts = dict(l1=l1.item(), lpips=lp.item(), kl=kl.item(), adv=0.0, lam=0.0)
    if disc is not None:
        g   = -disc(rec).mean()
        lam = adaptive_w(l_rec, g, vae.decoder.conv_out.weight)
        loss = loss + lam * w_adv * g
        parts.update(adv=g.item(), lam=float(lam))
    return loss, parts, rec

def seg_loss(logits, target):
    """BCE + soft Dice. BCE alone is dominated by the easy background pixels; Dice is
       scale-free in the object, which is the part we need to be right."""
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    num = 2 * (p * target).sum((1, 2, 3)) + 1
    den = p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1
    return bce + (1 - num / den).mean()

def seg_loss5(logits, target):
    """Cross-entropy + soft Dice for the 5-class map. Dice is averaged over CLASSES, so
       exposed root at 5.4% of the area weighs as much as bone at 36% -- and exposed root
       is the class the whole project is about."""
    ce = F.cross_entropy(logits, target)
    p = logits.softmax(1)
    t = F.one_hot(target, N_MASK_CLASSES).permute(0, 3, 1, 2).float()
    num = 2 * (p * t).sum((0, 2, 3)) + 1
    den = p.sum((0, 2, 3)) + t.sum((0, 2, 3)) + 1
    return ce + (1 - num / den).mean()

Metrics.

In [ ]:
def line_y(p1, p2):
    """Row of the line through two patch points, one value per column. Near-vertical pairs
       fall back to the mean row."""
    (x1, y1), (x2, y2) = p1, p2
    xs = np.arange(PATCH_W, dtype=np.float32)
    if abs(x2 - x1) < 5:
        return 0.5 * (y1 + y2) * np.ones_like(xs)
    return y1 + (y2 - y1) * (xs - x1) / (x2 - x1)

def crest_band(kp, half=16):
    """Boolean band of +-half px around the BL_M-BL_D line, in patch coords."""
    if kp[KP["BL_M"], 2] == 0 or kp[KP["BL_D"], 2] == 0:
        return None
    yl = line_y(kp[KP["BL_M"], :2], kp[KP["BL_D"], :2])
    ys = np.arange(PATCH_H, dtype=np.float32)[:, None]
    return np.abs(ys - yl[None, :]) <= half

def gradmag(a):
    """Sobel gradient magnitude. The crest is an edge, so losing it means losing gradient."""
    a = a.astype(np.float32)
    gx = cv2.Sobel(a, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(a, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx * gx + gy * gy)

def crest_ratio(orig, rec, kp, half=16):
    """Gradient-MAE in the crest band / over the whole patch. ~1.0 = no local bias."""
    band = crest_band(kp, half)
    if band is None or band.sum() < 50:
        return None
    e = np.abs(gradmag(orig) - gradmag(rec))
    return float(e[band].mean() / (e.mean() + 1e-8))

def hf_energy(a, s=1.5):
    """Std of the high-pass residual: energy at scales finer than s px.
       Trabecular bone lives at 1-3 px, so this is the texture the VAE tends to drop."""
    f = a.astype(np.float32)
    return float((f - cv2.GaussianBlur(f, (0, 0), s)).std())

def row_profile(a):
    """Mean intensity per row. Patches are crown-up, so this is the coronal-apical anatomy."""
    return a.astype(np.float32).mean(1)

def average_precision(y, p):
    """Area under precision-recall, step estimator."""
    o = np.argsort(-np.asarray(p, float)); y = np.asarray(y, int)[o]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    prec, rec = tp / (tp + fp), tp / max(y.sum(), 1)
    return float(np.sum(np.diff(np.concatenate([[0.0], rec])) * prec))

def macro_f1(y, p, thr=0.5):
    y, q = np.asarray(y, int), (np.asarray(p) >= thr).astype(int)
    f = []
    for c in (0, 1):
        tp = ((q == c) & (y == c)).sum()
        f.append(2 * tp / max(2 * tp + ((q == c) & (y != c)).sum() + ((q != c) & (y == c)).sum(), 1))
    return float(np.mean(f))

@torch.no_grad()
def fid_kid(real, gen, subset_size=50, subsets=50):
    """FID and KID between two lists of uint8 patches. At n~100 FID is biased upward
       (2048 features, fewer samples than dimensions); KID's estimator is unbiased."""
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
    kid = KernelInceptionDistance(subset_size=subset_size, subsets=subsets,
                                  normalize=False).to(DEVICE)
    for is_real, imgs in ((True, real), (False, gen)):
        x = torch.from_numpy(np.stack(imgs))[:, None].repeat(1, 3, 1, 1).to(DEVICE)
        fid.update(x, real=is_real)
        kid.update(x, real=is_real)
    km, ks = kid.compute()
    return float(fid.compute()), float(km), float(ks)

@torch.no_grad()
def lpips_diversity(imgs, n_pairs=200, seed=SEED):
    """Mean pairwise LPIPS over random pairs. Mode-collapse guard: a model that always
       draws the same tooth scores ~0 however good each sample looks."""
    rng = np.random.default_rng(seed)
    i, j = rng.integers(0, len(imgs), (2, n_pairs))
    k = i != j
    i, j = i[k], j[k]
    tot, n = 0.0, 0
    for s in range(0, len(i), 32):
        a = torch.stack([torch.from_numpy(norm_img(imgs[q]))[None] for q in i[s:s+32]]).to(DEVICE)
        b = torch.stack([torch.from_numpy(norm_img(imgs[q]))[None] for q in j[s:s+32]]).to(DEVICE)
        tot += perceptual(a, b).item() * len(a); n += len(a)
    return tot / n

Mask construction and its inverse.

In [ ]:
MASK_KPS = ("CEJ_M", "CEJ_D", "BL_M", "BL_D")

def apex_row(kp):
    """Apex row: the shortest root governs, so the
       most coronal visible root-level point sets the row."""
    v = [kp[j, 1] for j in (KP["RL_M"], KP["RL_D"], KP["RL_C"]) if kp[j, 2] > 0]
    return float(min(v)) if v else None

def build_mask(tooth, kp, fill=16):
    """5-class label map from a silhouette and the keypoints. The CEJ and crest lines cut
       the tooth into crown / exposed root / embedded root; class 4 is the bone apical to
       the crest. None if a landmark is unavailable."""
    if any(kp[KP[k], 2] == 0 for k in MASK_KPS):
        return None
    y_ap = apex_row(kp)
    if y_ap is None:
        return None

    ys = np.arange(PATCH_H, dtype=np.float32)[:, None]
    t = tooth.astype(bool) & (ys <= y_ap)
    if fill:
        r = int(y_ap)
        for x in range(PATCH_W):
            col = np.where(t[:, x])[0]
            if len(col) and 0 < r - col[-1] <= fill:
                t[col[-1]:r + 1, x] = True

    below_cej   = ys > line_y(kp[KP["CEJ_M"], :2], kp[KP["CEJ_D"], :2])[None, :]
    below_crest = ys > line_y(kp[KP["BL_M"], :2], kp[KP["BL_D"], :2])[None, :]
    m = np.zeros((PATCH_H, PATCH_W), np.uint8)
    m[t & ~below_cej]               = 1     # crown
    m[t & below_cej & ~below_crest] = 2     # exposed root <- the bone-loss signature
    m[t & below_crest]              = 3     # embedded root
    m[~t & below_crest]             = 4     # alveolar bone
    return m

def realized_pbl(mask):
    """Read PBL back out of a 5-class map, so a commanded mask can be scored the same way
       a real one is, and so S_eval's output can be scored at all. Per column:
       (crest - CEJ) / (apex - CEJ), then the median over columns.
       An empty class 2 means the crest sits at the CEJ, i.e. PBL 0."""
    out = []
    for x in range(PATCH_W):
        col = mask[:, x]
        c1, c2, c3 = np.where(col == 1)[0], np.where(col == 2)[0], np.where(col == 3)[0]
        if len(c1) == 0 or len(c3) == 0:
            continue
        y_cej, y_apex = c1[-1], c3[-1]
        y_crest = c2[-1] if len(c2) else y_cej
        if y_apex - y_cej < 5:
            continue
        out.append((y_crest - y_cej) / (y_apex - y_cej))
    return float(np.median(out)) if out else None

def dice_per_class(pred, target):
    """Per-class Dice for one mask. NaN where a class is absent from both, so the mean over
       images skips it instead of awarding a free 1.0."""
    out = np.full(N_MASK_CLASSES, np.nan)
    for k in range(N_MASK_CLASSES):
        a, b = pred == k, target == k
        s = a.sum() + b.sum()
        if s:
            out[k] = 2 * (a & b).sum() / s
    return out

def retarget_mask(m, pbl_target):
    """Move the crest to a commanded PBL, holding the tooth silhouette fixed. This is the
       augmentation engine: generating a Severe case does not require having seen many
       Severe cases, only a mask whose crest sits low on an otherwise ordinary tooth.
       Per column, because y_cej and y_apex already vary per column."""
    out = m.copy()
    tooth = (m >= 1) & (m <= 3)
    rows = np.arange(PATCH_H)
    for x in range(PATCH_W):
        t, c1 = tooth[:, x], np.where(m[:, x] == 1)[0]
        if not t.any() or len(c1) == 0:
            continue
        y_cej, y_apex = c1[-1], np.where(t)[0][-1]
        if y_apex - y_cej < 5:
            continue
        below = rows > y_cej + pbl_target * (y_apex - y_cej)
        out[~below & ~t, x] = 0
        out[(rows <= y_cej) & t, x] = 1
        out[(rows > y_cej) & ~below & t, x] = 2     # exposed root grows as the crest drops
        out[below & t, x] = 3
        out[below & ~t, x] = 4
    return out

Conditioning.

In [ ]:
def mask_ablate(m, p_class=0.25, p_null=0.1):
    """Konz Algorithm 1: per sample, per class c in 1..4 independently, erase c with
       probability p_class."""
    m = m.clone()
    for i in range(len(m)):
        if random.random() < p_null:
            m[i] = 0
            continue
        for c in range(1, N_MASK_CLASSES):
            if random.random() < p_class:
                m[i][m[i] == c] = 0
    return m

def cond_channels(m, hw=(LATENT_H, LATENT_W)):
    """Integer map at 256x128 -> one-hot float at hw. hw is the latent size for the LDM and
       the full patch size for the pixel-space baseline, so both models get the identical
       conditioning construction and only the resolution differs."""
    small = F.interpolate(m[:, None].float(), size=hw, mode="nearest")
    return F.one_hot(small[:, 0].long(), N_MASK_CLASSES).permute(0, 3, 1, 2).float()

# Data

Raw radiographs.

In [ ]:
rng = np.random.default_rng(SEED)
picks = rng.choice(STEMS, 8, replace=False)
imgs = [read_img(BASE_IMG / f"{s}.png") for s in picks]
show_grid(imgs, [f"{s}\n{i.shape[1]}x{i.shape[0]}" for s, i in zip(picks, imgs)],
          ncols=4, save="d1_raw.png")

sizes = collections.Counter(read_img(BASE_IMG / f"{s}.png").shape for s in STEMS)
print(f"{len(STEMS)} radiographs, {len(sizes)} distinct sizes")
print("most common:", sizes.most_common(3))
by_area = sorted(sizes, key=lambda hw: hw[0] * hw[1])
print(f"smallest {by_area[0][1]}x{by_area[0][0]}   largest {by_area[-1][1]}x{by_area[-1][0]}  (WxH)")


Annotations: box class, rotating box, 11 keypoints.

In [ ]:
picks = ["Image100", "Image108", "Image38", "Image5"]
imgs, titles = [], []
for s in picks:
    g = read_img(BASE_IMG / f"{s}.png"); H, W = g.shape
    objs, angs = read_pose_label(BASE_LAB / f"{s}.txt"), read_rot_angles(BASE_ROT / f"{s}.txt")
    canvas, kinds = to_canvas(g), []
    for o, a in zip(objs, angs):
        if o["cls"] not in TOOTH_CLASSES:
            continue
        kps = denorm_kps(o["kps"], W, H)
        draw_box(canvas, denorm_box(o["box"], W, H), a)
        draw_axis(canvas, kps, denorm_box(o["box"], W, H), a)
        draw_kps(canvas, kps)
        kinds.append(BOX_CLASSES[o["cls"]][:6])
    imgs.append(canvas); titles.append(f"{s}  {W}x{H}\n{', '.join(kinds)}")

fig = show_grid(imgs, titles, ncols=2, size=5.5)
fig.legend(handles=[Line2D([], [], marker="o", ls="", markersize=6,
                           color=np.array(KP_COLORS[n])/255, label=n) for n in KP],
           loc="lower center", ncol=6, fontsize=7, frameon=False)
fig.savefig(FIGS / "d2_annotations.png", dpi=130, bbox_inches="tight")


Auxiliary segmentation set: pseudo-periapical crops of panoramic radiographs with tooth polygons.

In [ ]:
AUX_STEMS = sorted(p.stem for p in (AUX_TR / "images").glob("*.JPG"))
rng = np.random.default_rng(SEED)
picks = rng.choice(AUX_STEMS, 6, replace=False)

imgs, titles = [], []
for s in picks:
    g = read_img(AUX_TR / "images" / f"{s}.JPG"); H, W = g.shape
    polys = [p * (W, H) for p in read_seg_label(AUX_TR / "labels" / f"{s}.txt")]
    imgs.append(draw_polys(to_canvas(g), polys))
    titles.append(f"{s}  {W}x{H}  {len(polys)} teeth")
show_grid(imgs, titles, ncols=3, save="d3_auxseg.png")

Build tooth records.

In [ ]:
TEETH, DROPPED = scan_teeth()

n_sides = sum(p is not None and p <= PBL_MAX for t in TEETH for p in (t["pbl_m"], t["pbl_d"]))
hist = collections.Counter(t["stage"] for t in TEETH)
img_adv = {t["stem"] for t in TEETH if t["pbl"] >= ADVANCED_MIN}
img_str = {t["stem"] for t in TEETH if t["pbl"] >= SEVERE_MIN}

print(f"sides kept {n_sides} - dropped {len(DROPPED)} - teeth {len(TEETH)}")
for s, name in enumerate(STAGE_NAMES):
    print(f"  {name} {hist[s]}  ({hist[s]/len(TEETH)*100:4.1f}%)")
print(f"Advanced (>= {ADVANCED_MIN}) teeth {sum(t['advanced'] for t in TEETH)} "
      f"| images {len(img_adv)}   (clinicians: 70)")
print(f"severe   (>= {SEVERE_MIN}) teeth {len(img_str)}")
print("dropped:", [(s, sd, round(p, 2)) for s, sd, p in DROPPED])

Statistics.

In [ ]:
cls_n, per_img = collections.Counter(), []
for s in STEMS:
    objs = read_pose_label(BASE_LAB / f"{s}.txt")
    for o in objs:
        cls_n[o["cls"]] += 1
    per_img.append(sum(o["cls"] in TOOTH_CLASSES for o in objs))

fig, ax = plt.subplots(2, 2, figsize=(11, 7))

ax[0,0].bar(range(5), [cls_n[i] for i in range(5)], color="steelblue")
ax[0,0].set_xticks(range(5)); ax[0,0].set_xticklabels(BOX_CLASSES, rotation=20, fontsize=7)
ax[0,0].set_title(f"box classes (n={sum(cls_n.values())})", fontsize=9)
for i in range(5):
    ax[0,0].text(i, cls_n[i], str(cls_n[i]), ha="center", va="bottom", fontsize=7)

c = collections.Counter(per_img)
ax[0,1].bar(sorted(c), [c[k] for k in sorted(c)], color="seagreen")
ax[0,1].set_title(f"teeth per radiograph (total {sum(per_img)})", fontsize=9)
ax[0,1].set_xlabel("teeth"); ax[0,1].set_ylabel("radiographs")

sides = np.array([p for t in TEETH for p in (t["pbl_m"], t["pbl_d"])
                  if p is not None and p <= PBL_MAX])
ax[1,0].hist(sides, bins=50, color="darkorange")
for e in STAGE_EDGES:
    ax[1,0].axvline(e, color="k", ls="--", lw=1)
ax[1,0].set_title(f"PBL per tooth-side (n={len(sides)})", fontsize=9); ax[1,0].set_xlabel("PBL")
for e, n in zip((0.0,) + STAGE_EDGES, STAGE_NAMES):
    ax[1,0].text(e + 0.005, ax[1,0].get_ylim()[1] * 0.93, n, fontsize=7, rotation=90, va="top")

hist = collections.Counter(t["stage"] for t in TEETH)
bars = ax[1,1].bar(range(4), [hist[i] for i in range(4)],
                   color=["#4c9f70", "#d9c14e", "#e08b3c", "#c1362f"])
ax[1,1].set_xticks(range(4)); ax[1,1].set_xticklabels(STAGE_NAMES, fontsize=8)
ax[1,1].set_title(f"tooth severity, worst side (n={len(TEETH)})", fontsize=9)
for i, b in enumerate(bars):
    ax[1,1].text(b.get_x() + b.get_width()/2, hist[i],
                 f"{hist[i]}\n{hist[i]/len(TEETH)*100:.1f}%", ha="center", va="bottom", fontsize=7)
n_adv = sum(t["pbl"] >= ADVANCED_MIN for t in TEETH)
n_sev = sum(t["pbl"] >= SEVERE_MIN  for t in TEETH)
ax[1,1].set_xlabel(f"Advanced (>={ADVANCED_MIN}) {n_adv} teeth   |   Severe (>={SEVERE_MIN}) {n_sev}",
                   fontsize=8)

fig.tight_layout(); fig.savefig(FIGS / "d6_stats.png", dpi=130, bbox_inches="tight")


Build the patches.

In [ ]:
PATCHES = []
for t in tqdm(TEETH, desc="patches"):
    img = read_img(BASE_IMG / f"{t['stem']}.png")
    patch, kp, meta = extract_patch(img, t["box"], t["angle"], t["kps"])
    PATCHES.append({**t, "patch": patch, "kp": kp, "meta": meta})

cej_apex = [crown_apex(p["kp"]) for p in PATCHES]
dy    = np.array([a[1] - c[1] for c, a in cej_apex])
ratio = np.array([abs(a[0] - c[0]) / max(abs(a[1] - c[1]), 1e-6) for c, a in cej_apex])
print(f"patches {len(PATCHES)} | flipped {sum(p['meta']['flipped'] for p in PATCHES)}")
print(f"apex below CEJ: {(dy > 0).sum()}/{len(dy)}")
print(f"|dx|/|dy|: median {np.median(ratio):.4f}  p95 {np.percentile(ratio, 95):.4f}")
assert (dy > 0).all(), "crown-up canonicalisation failed"
assert np.median(ratio) < 0.10, "teeth are not vertical after rotation"

In [ ]:
order = np.argsort([p["pbl"] for p in PATCHES])
sel   = [PATCHES[order[i]] for i in np.linspace(0, len(order) - 1, 16).astype(int)]

imgs, titles = [], []
for p in sel:
    c = to_canvas(p["patch"])
    draw_kps(c, p["kp"], only=("CEJ_M", "CEJ_D", "BL_M", "BL_D", "RL_C", "RL_M", "RL_D"))
    imgs.append(c)
    titles.append(f'{p["stem"]}#{p["idx"]}  PBL {p["pbl"]:.2f}\n'
                  f'{STAGE_NAMES[p["stage"]]}  flip={p["meta"]["flipped"]}')

show_grid(imgs, titles, ncols=8, size=2.2, save="d7_patches.png")

Auxiliary tooth patches extracted in identical geometry.

In [ ]:
AUX_MIN_AREA, AUX_MIN_ELONG, AUX_BORDER = 500.0, 1.3, 3.0

def build_aux_patches(split_dir, limit=None):
    """Aux-seg tooth patches in the SAME geometry as the real ones, plus GT silhouettes."""
    stems = sorted(p.stem for p in (split_dir / "images").glob("*.JPG"))
    patches, masks = [], []
    stats = collections.Counter()
    for s in tqdm(stems, desc=split_dir.name):
        img = read_img(split_dir / "images" / f"{s}.JPG"); H, W = img.shape
        for poly in read_seg_label(split_dir / "labels" / f"{s}.txt"):
            P = poly * (W, H); stats["total"] += 1
            if cv2.contourArea(P.astype(np.float32)) < AUX_MIN_AREA:
                stats["small"] += 1; continue
            if (P[:, 0].min() < AUX_BORDER or P[:, 1].min() < AUX_BORDER or
                P[:, 0].max() > W - AUX_BORDER or P[:, 1].max() > H - AUX_BORDER):
                stats["border"] += 1; continue
            box, th = poly_box_axis(P)
            if box[3] / max(box[2], 1e-6) < AUX_MIN_ELONG:
                stats["squat"] += 1; continue
            patch, _, meta = extract_patch(img, box, th, np.zeros((N_KP, 3)))
            m = np.zeros((PATCH_H, PATCH_W), np.uint8)
            cv2.fillPoly(m, [warp_points(P, meta).astype(np.int32)], 1)
            patches.append(patch); masks.append(m); stats["kept"] += 1
            if limit and stats["kept"] >= limit:
                return np.array(patches), np.array(masks), stats
    return np.array(patches), np.array(masks), stats

AUX_NPZ = CACHE / "aux_patches.npz"
if AUX_NPZ.exists():
    d = np.load(AUX_NPZ)
    AUX_X, AUX_Y, AUX_XV, AUX_YV = d["x"], d["y"], d["xv"], d["yv"]
    aux_stats = {"cached": len(AUX_X)}
else:
    AUX_X, AUX_Y, aux_stats = build_aux_patches(AUX_TR)
    AUX_XV, AUX_YV, _       = build_aux_patches(AUX_VA)
    np.savez_compressed(AUX_NPZ, x=AUX_X, y=AUX_Y, xv=AUX_XV, yv=AUX_YV)


if "kept" in aux_stats:
    print(dict(aux_stats), f"-> kept {aux_stats['kept']/aux_stats['total']*100:.0f}%")
print("train", AUX_X.shape, AUX_Y.shape, "| val", AUX_XV.shape)
print(f"mask fill fraction: mean {AUX_Y.mean():.3f}  (real around 0.40)")



In [ ]:
rng  = np.random.default_rng(SEED)
sel  = rng.choice(len(AUX_X), 12, replace=False)
TOOTH_COLORS = ((0, 0, 0), (255, 90, 90))

imgs, titles = [], []
for i in sel:
    raw = to_canvas(AUX_X[i])
    imgs.append(np.hstack([raw, overlay_mask(raw, AUX_Y[i], TOOTH_COLORS)]))
    titles.append(f"#{i}  patch | + silhouette   fill {AUX_Y[i].mean():.2f}")

show_grid(imgs, titles, ncols=6, size=2.6, save="d8_aux_patches.png")


Split into train / val / test by image, instead of by tooth: two teeth from one radiograph share exposure, angle and patient, so a tooth-level split would leak.

In [ ]:
SPLITS_JSON = CACHE / "splits.json"

if SPLITS_JSON.exists():
    SPLITS = json.loads(SPLITS_JSON.read_text())
else:
    # image-level and stratified
    img_pbl = collections.defaultdict(float)
    for t in TEETH:
        img_pbl[t["stem"]] = max(img_pbl[t["stem"]], t["pbl"])
    by_bin = collections.defaultdict(list)
    for s in STEMS:
        by_bin[stage_of(img_pbl.get(s, 0.0))].append(s)

    frac = (0.7, 0.1, 0.2)
    rng, out = np.random.default_rng(SEED), [[], [], []]
    for b in sorted(by_bin):
        g = np.array(sorted(by_bin[b])); rng.shuffle(g)
        cuts = [round(len(g) * frac[0]), round(len(g) * (frac[0] + frac[1]))]
        for dst, part in zip(out, np.split(g, cuts)):
            dst += list(part)
    SPLITS = {k: sorted(v) for k, v in zip(("train", "val", "test"), out)}
    SPLITS_JSON.write_text(json.dumps(SPLITS, indent=1))

sets = [set(SPLITS[k]) for k in ("train", "val", "test")]
assert set.union(*sets) == set(STEMS), "fewer images than expected"
assert sum(map(len, sets)) == len(STEMS), "an image is in two splits"

SPLIT_OF = {s: k for k in SPLITS for s in SPLITS[k]}
for p in PATCHES:
    p["split"] = SPLIT_OF[p["stem"]]

for k in ("train", "val", "test"):
    T = [p for p in PATCHES if p["split"] == k]
    h = collections.Counter(p["stage"] for p in T)
    adv = sum(p["pbl"] >= ADVANCED_MIN for p in T)
    print(f"{k:5s} img {len(SPLITS[k]):3d}  patches {len(T):3d} | "
          + " ".join(f"{STAGE_NAMES[i][:4]} {h[i]:3d}" for i in range(4))
          + f" | Advanced {adv:3d} ({adv/len(T)*100:4.1f}%)")

Datasets + rigid only jitter (flip, small rotation, translation)

In [ ]:
class PatchDataset(torch.utils.data.Dataset):
    """128x256 patches in [-1,1]. Returns the image alone when y is None (VAE, LDM),
       or (image, mask) when y is given (S_build, S_eval).
       vflip only for aux patches since real ones are canonicalized crown-up."""
    def __init__(self, x, y=None, train=True, vflip=False, affine=False):
        self.x, self.y, self.train, self.vflip, self.affine = x, y, train, vflip, affine

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = self.x[i].astype(np.float32)
        msk = None if self.y is None else self.y[i]
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1]
                if msk is not None: msk = msk[:, ::-1]
            if self.vflip and random.random() < 0.5:
                img = img[::-1]
                if msk is not None: msk = msk[::-1]
            if self.affine:
                # rigid jitter only: no rescale, so PBL is unchanged
                h, w = img.shape
                M = cv2.getRotationMatrix2D((w/2, h/2), random.uniform(-5, 5), 1)
                M[0, 2] += random.uniform(-0.04, 0.04) * w
                M[1, 2] += random.uniform(-0.04, 0.04) * h
                img = cv2.warpAffine(np.ascontiguousarray(img), M, (w, h),
                                     flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
                if msk is not None:
                    msk = cv2.warpAffine(np.ascontiguousarray(msk), M, (w, h),
                                         flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REPLICATE)
            img = img * random.uniform(0.85, 1.15) + random.uniform(-15, 15)
        img = torch.from_numpy(np.ascontiguousarray(norm_img(np.clip(img, 0, 255))))[None]
        if msk is None:
            return img
        return img, torch.from_numpy(np.ascontiguousarray(msk)).long()

class ClsDataset(torch.utils.data.Dataset):
    """Patches with a binary label. Same flip and brightness jitter as everywhere else; no
       affine, because there is no mask to keep in sync and no PBL to preserve here."""
    def __init__(self, x, y, train=True):
        self.x, self.y, self.train = x, np.asarray(y, np.int64), train

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = self.x[i].astype(np.float32)
        if self.train:
            if random.random() < 0.5:
                img = img[:, ::-1]
            img = img * random.uniform(0.85, 1.15) + random.uniform(-15, 15)
        img = torch.from_numpy(np.ascontiguousarray(norm_img(np.clip(img, 0, 255))))[None]
        return img, int(self.y[i])

real_tr = np.stack([p["patch"] for p in PATCHES if p["split"] == "train"])
real_va = np.stack([p["patch"] for p in PATCHES if p["split"] == "val"])
VAE_X   = np.concatenate([real_tr, AUX_X])      # 395 real + 8977 aux

w = np.concatenate([np.full(len(real_tr), len(AUX_X) / len(real_tr)), np.ones(len(AUX_X))])
vae_sampler = torch.utils.data.WeightedRandomSampler(torch.from_numpy(w), len(VAE_X), replacement=True)

set_seed()
vae_dl  = torch.utils.data.DataLoader(PatchDataset(VAE_X, train=True), 16,
                                      sampler=vae_sampler, drop_last=True)
vae_dlv = torch.utils.data.DataLoader(PatchDataset(real_va, train=False), 16)

print(f"VAE train {len(VAE_X)} ({len(real_tr)} real + {len(AUX_X)} aux), resampled to ~50% real "
      f"| val {len(real_va)} real")

In [ ]:
set_seed()
build_dl  = torch.utils.data.DataLoader(PatchDataset(AUX_X, AUX_Y, train=True, vflip=True),
                                        32, shuffle=True, drop_last=True)
build_dlv = torch.utils.data.DataLoader(PatchDataset(AUX_XV, AUX_YV, train=False), 32)
xb, yb = next(iter(build_dl))
print(xb.shape, xb.dtype, f"[{xb.min():.2f}, {xb.max():.2f}]", "|", yb.shape, yb.dtype, yb.unique().tolist())

# LDM trains on real patches only, with the rigid jitter on
ldm_dl = torch.utils.data.DataLoader(PatchDataset(real_tr, train=True, affine=True),
                                     32, shuffle=True, drop_last=True)

# Network

## VAE

AutoencoderKL, f=4, 4-channel latent. Small due to hardware limitations.

In [ ]:
def make_vae(ch=(32, 64, 128), ckpt=True):
    """f=4 (3 blocks -> 2 downsamples): 256x128 -> 64x32x4."""
    vae = AutoencoderKL(
        in_channels=1, out_channels=1, latent_channels=LATENT_CH,
        block_out_channels=ch, layers_per_block=2,
        down_block_types=("DownEncoderBlock2D",) * len(ch),
        up_block_types=("UpDecoderBlock2D",) * len(ch),
        norm_num_groups=16,
    )
    if ckpt:
        vae.enable_gradient_checkpointing()
    return vae

set_seed()
vae = make_vae().to(DEVICE)
z = vae.encode(torch.zeros(2, 1, PATCH_H, PATCH_W, device=DEVICE)).latent_dist.mean
assert z.shape[1:] == (LATENT_CH, LATENT_H, LATENT_W), z.shape
print(f"{sum(p.numel() for p in vae.parameters())/1e6:.2f}M params | latent {tuple(z.shape[1:])} | f={PATCH_H // z.shape[2]}")


PatchGAN (Isola et al., 2017) with hinge loss, used for the VAE's adversarial term.

In [ ]:
class PatchDisc(nn.Module):
    def __init__(self, ch=64, n_layers=3):
        super().__init__()
        layers, c = [nn.Conv2d(1, ch, 4, 2, 1), nn.LeakyReLU(0.2, True)], ch
        for k in range(1, n_layers):
            c2 = min(ch * 2 ** k, 256)
            layers += [nn.Conv2d(c, c2, 4, 2, 1, bias=False),
                       nn.GroupNorm(8, c2),          # not BatchNorm: no batch statistics
                       nn.LeakyReLU(0.2, True)]
            c = c2
        layers += [nn.Conv2d(c, 1, 4, 1, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

set_seed()
disc = PatchDisc().to(DEVICE)
o = disc(torch.zeros(2, 1, PATCH_H, PATCH_W, device=DEVICE))
print(f"{sum(p.numel() for p in disc.parameters())/1e6:.2f}M params | score map {tuple(o.shape[1:])}")

## LDM

Same scheduler for every diffusion model: 1000 steps, cosine betas, epsilon prediction.

In [ ]:
sched = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2",
                      prediction_type="epsilon")

UNet on the latent.

In [ ]:
def make_unet(ch=(128, 256, 256), mask_ch=0):
    """Denoiser on 4x64x32 latents. mask_ch=0 here (unconditional baseline);
       Phase B calls the same function with mask_ch=N_MASK_CLASSES for the
       channel-concatenated anatomical mask."""
    return UNet2DModel(
        sample_size=(LATENT_H, LATENT_W),
        in_channels=LATENT_CH + mask_ch,
        out_channels=LATENT_CH,                  # predicts epsilon, same shape as the latent
        block_out_channels=ch,
        layers_per_block=2,
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
        attention_head_dim=32,
        norm_num_groups=32,
        resnet_time_scale_shift="scale_shift",   # AdaGN: modulate AFTER the norm, not before
    )

set_seed()
unet = make_unet().to(DEVICE)

z = torch.zeros(2, LATENT_CH, LATENT_H, LATENT_W, device=DEVICE)
t = torch.zeros(2, dtype=torch.long, device=DEVICE)
assert unet(z, t).sample.shape == z.shape, "the denoiser must return a latent, not an image"

sampler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2",
                                    prediction_type="epsilon", set_alpha_to_one=False)

print(f"{sum(p.numel() for p in unet.parameters())/1e6:.1f}M params | "
      f"{LATENT_CH}ch -> {LATENT_CH}ch @ {LATENT_H}x{LATENT_W}")
print(f"train: {type(sched).__name__} 1000 steps | sample: {type(sampler).__name__} ~25 steps")


## DDPM

Pixel baseline following Konz et al., almost equal parameter count as the LDM.

In [ ]:
def make_unet_px(ch=(64, 96, 128, 256, 256), mask_ch=N_MASK_CLASSES):
    n = len(ch)
    return UNet2DModel(
        sample_size=(PATCH_H, PATCH_W),
        in_channels=1 + mask_ch, out_channels=1,
        block_out_channels=ch, layers_per_block=2,
        down_block_types=("DownBlock2D",) * (n - 2) + ("AttnDownBlock2D",) * 2,
        up_block_types=("AttnUpBlock2D",) * 2 + ("UpBlock2D",) * (n - 2),
        attention_head_dim=32, norm_num_groups=32,
        resnet_time_scale_shift="scale_shift")

set_seed()
unet_px = make_unet_px().to(DEVICE)
o = unet_px(torch.zeros(1, 1 + N_MASK_CLASSES, PATCH_H, PATCH_W, device=DEVICE),
            torch.zeros(1, dtype=torch.long, device=DEVICE)).sample
assert o.shape[1:] == (1, PATCH_H, PATCH_W), o.shape
print(f"{sum(p.numel() for p in unet_px.parameters())/1e6:.2f}M params | "
      f"1+{N_MASK_CLASSES}ch -> 1ch @ {PATCH_H}x{PATCH_W}   (LDM: 29.1M @ {LATENT_H}x{LATENT_W})")
del unet_px, o; torch.cuda.empty_cache()

## S_build

4-level UNet to build the tooth silhouette from auxiliary data (out_ch=1) or to evaluate (out_ch=5).

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(True))

class SegUNet(nn.Module):
    """Plain 4-level UNet. out_ch=1 for S_build (tooth silhouette), 5 for S_eval later."""
    def __init__(self, out_ch, base=32):
        super().__init__()
        ch = [base * 2 ** k for k in range(4)]                       # 32 64 128 256
        self.enc  = nn.ModuleList([conv_block(1 if k == 0 else ch[k-1], ch[k]) for k in range(4)])
        self.bott = conv_block(ch[-1], ch[-1] * 2)
        self.up   = nn.ModuleList([nn.ConvTranspose2d(ch[k] * 2, ch[k], 2, 2) for k in range(4)])
        self.dec  = nn.ModuleList([conv_block(ch[k] * 2, ch[k]) for k in range(4)])
        self.head = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.bott(x)
        for k in reversed(range(len(skips))):
            x = self.dec[k](torch.cat([self.up[k](x), skips[k]], 1))
        return self.head(x)

set_seed()
s_build = SegUNet(out_ch=1).to(DEVICE)
o = s_build(torch.zeros(2, 1, PATCH_H, PATCH_W, device=DEVICE))
assert o.shape[1:] == (1, PATCH_H, PATCH_W), o.shape
print(f"{sum(p.numel() for p in s_build.parameters())/1e6:.2f}M params | {tuple(o.shape[1:])}")

## ResNet-18 Classifier

In [ ]:
def make_clf():
    """ResNet-18 with ImageNet weights, conv1 collapsed to one channel by summing the RGB."""
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    w = m.conv1.weight.data.sum(1, keepdim=True)
    m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    m.conv1.weight.data = w
    m.fc = nn.Linear(512, 2)
    return m

set_seed()
clf = make_clf().to(DEVICE)
o = clf(torch.zeros(2, 1, PATCH_H, PATCH_W, device=DEVICE))
assert o.shape == (2, 2), o.shape
print(f"{sum(p.numel() for p in clf.parameters())/1e6:.2f}M params | 1ch {PATCH_H}x{PATCH_W} -> 2")
del clf, o; torch.cuda.empty_cache()

# Train

## VAE  

In [ ]:
@torch.no_grad()
def eval_vae(vae, dl):
    """L1, LPIPS and high-frequency retention on real VAL patches."""
    vae.eval(); tot, n = collections.Counter(), 0
    for x in dl:
        x = x.to(DEVICE)
        with torch.autocast(DEVICE, dtype=DTYPE):
            rec = vae(x).sample                    # sample_posterior=False -> the mode
        tot["l1"]    += (rec.float() - x).abs().mean().item() * len(x)
        tot["lpips"] += perceptual(rec.float(), x).item() * len(x)
        for a, b in zip(x[:, 0].cpu().numpy(), rec[:, 0].float().cpu().numpy()):
            tot["hf"] += hf_energy(denorm_img(b)) / (hf_energy(denorm_img(a)) + 1e-8)
        n += len(x)
    return {k: v / n for k, v in tot.items()}

def train_vae(vae, disc, dl, dlv, steps=12000, lr=1e-4, val_every=500, disc_start=3000):
    opt   = torch.optim.AdamW(vae.parameters(), lr)
    opt_d = torch.optim.AdamW(disc.parameters(), 4 * lr, betas=(0.5, 0.9))
    lr_g  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    hist  = dict(disc_start=disc_start, step=[], l1=[], lpips=[], adv=[],
                 lam=[], d=[], sep=[],
                 val_step=[], val_l1=[], val_lpips=[], val_hf=[])
    best, it = float("inf"), 0
    pbar = tqdm(total=steps, desc="vae")
    while it < steps:
        for x in dl:
            if it >= steps:
                break
            on = it >= disc_start
            vae.train(); disc.train()
            x = x.to(DEVICE, non_blocking=True)

            with torch.autocast(DEVICE, dtype=DTYPE):
                loss, parts, rec = vae_loss(vae, x, disc if on else None)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            opt.step(); opt.zero_grad(set_to_none=True); lr_g.step()

            d_loss, sep = 0.0, 0.0
            if on:
                with torch.autocast(DEVICE, dtype=DTYPE):
                    s_real, s_fake = disc(x), disc(rec.detach())
                    d = hinge_d(s_real, s_fake)
                d.backward()
                opt_d.step(); opt_d.zero_grad(set_to_none=True)
                d_loss = d.item()
                sep = (s_real.mean() - s_fake.mean()).item()

            it += 1; pbar.update(1)
            hist["step"].append(it); hist["l1"].append(parts["l1"])
            hist["lpips"].append(parts["lpips"]); hist["adv"].append(parts["adv"])
            hist["lam"].append(parts["lam"]); hist["d"].append(d_loss); hist["sep"].append(sep)

            if it % val_every == 0 or it == steps:
                v = eval_vae(vae, dlv)
                hist["val_step"].append(it); hist["val_l1"].append(v["l1"])
                hist["val_lpips"].append(v["lpips"]); hist["val_hf"].append(v["hf"])
                if on and v["lpips"] < best:
                    best = v["lpips"]
                    torch.save(vae.state_dict(), CKPT / "vae.pt")
                    torch.save(disc.state_dict(), CKPT / "disc.pt")
                pbar.set_postfix(l1=f"{parts['l1']:.4f}", lam=f"{parts['lam']:.1f}",
                                 d=f"{d_loss:.3f}", sep=f"{sep:+.2f}",
                                 val_lpips=f"{v['lpips']:.3f}", val_hf=f"{v['hf']:.3f}",
                                 best=f"{best:.3f}")
    pbar.close()
    print(f"best val LPIPS {best:.4f} | peak VRAM {torch.cuda.max_memory_allocated()/2**20:.0f} MiB")
    VAE_META.write_text(json.dumps({"hist": hist, "best_val_lpips": best,
                                    "steps": steps, "disc_start": disc_start}))
    return hist

set_seed()
vae  = make_vae().to(DEVICE)
disc = PatchDisc().to(DEVICE)

if (CKPT / "vae.pt").exists():
    hist_vae = json.loads(VAE_META.read_text()).get("hist")
    print("VAE: cached checkpoint, training skipped (delete cache/ckpt/vae.pt to retrain)")
else:
    hist_vae = train_vae(vae, disc, vae_dl, vae_dlv, steps=12000)

vae.load_state_dict(torch.load(CKPT / "vae.pt"))
vae.eval()
for p in vae.parameters():
    p.requires_grad_(False) # frozen

In [ ]:
# baseline: the same curve from the run with uniform sampling, where the discriminator
# collapsed to a constant and contributed nothing
HF_BASELINE = [0.252, 0.293, 0.357, 0.403, 0.425, 0.442, 0.407, 0.454, 0.500, 0.447, 0.523, 0.519,
               0.550, 0.539, 0.541, 0.547, 0.554, 0.563, 0.567, 0.558, 0.568, 0.565, 0.569, 0.569]

if hist_vae is not None:
    h = hist_vae
    k = max(1, len(h["step"]) // 300)
    ds = h.get("disc_start", 3000)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))

    ax[0].plot(h["step"][::k], h["l1"][::k], lw=0.7, color="steelblue", label="train L1")
    ax[0].plot(h["val_step"], h["val_l1"], "o-", ms=3, color="darkorange", label="val L1")
    ax[0].set_title("reconstruction (L1 rises when the GAN starts: expected)", fontsize=9)

    ax[1].plot(h["val_step"], h["val_hf"], "o-", ms=3, color="crimson", label="val HF retained")
    ax[1].plot(h["val_step"][:len(HF_BASELINE)], HF_BASELINE[:len(h["val_step"])],
               "s--", ms=3, color="grey", label="uniform sampling, dead disc")
    ax[1].plot(h["val_step"], h["val_lpips"], "o-", ms=3, color="seagreen", label="val LPIPS")
    ax[1].axhline(0.75, color="crimson", ls=":", lw=1)
    ax[1].set_title("Gate 2b, against the previous run", fontsize=9)

    ax[2].plot(h["step"][::k], h["d"][::k], lw=0.7, color="purple", label="disc hinge")
    ax[2].plot(h["step"][::k], h["sep"][::k], lw=0.7, color="black", label="score(real)-score(fake)")
    ax[2].axhline(1.0, color="purple", ls=":", lw=1)
    ax[2].axhline(0.0, color="black", ls=":", lw=1)
    ax[2].set_title("adversarial game (sep ~ 0 means the disc is dead)", fontsize=9)

    for a in ax:
        a.axvline(ds, color="k", ls="--", lw=1); a.set_xlabel("step"); a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIGS / "a7_vae_train.png", dpi=130, bbox_inches="tight")

## LDM

In [ ]:
real_dl = torch.utils.data.DataLoader(PatchDataset(real_tr, train=False), 16)
meta = json.loads(VAE_META.read_text())

if "latent_scale" not in meta:
    zs = []
    for x in real_dl:
        with torch.no_grad(), torch.autocast(DEVICE, dtype=DTYPE):
            zs.append(vae.encode(x.to(DEVICE)).latent_dist.mean.float().cpu())
    meta["latent_scale"] = float(1.0 / torch.cat(zs).std())
    VAE_META.write_text(json.dumps(meta))
LATENT_SCALE = meta["latent_scale"]

print(f"LATENT_SCALE = {LATENT_SCALE:.4f}")

In [ ]:
set_seed()
x = torch.from_numpy(norm_img(real_tr[0]))[None, None].to(DEVICE)
with torch.no_grad():
    z0 = vae.encode(x).latent_dist.mean * LATENT_SCALE

noise = torch.randn_like(z0)
imgs, titles = [], []
for t in (0, 100, 300, 600, 999):
    zt = sched.add_noise(z0, noise, torch.tensor([t]))
    with torch.no_grad():
        rec = vae.decode(zt / LATENT_SCALE).sample
    imgs.append(denorm_img(rec[0, 0].float().cpu().numpy()))
    titles.append(f"t={t}  signal {sched.alphas_cumprod[t].sqrt():.3f}")

show_grid(imgs, titles, ncols=5, size=2.6, save="a2_forward_process.png")

Anatomical loss: the anatomical term decodes the x_0 prediction, pushes it through the frozen S_eval and penalizes disagreement with the mask that was actually commanded.

In [ ]:
def anat_term(seg, pred, zt, t, m_true, m_used, latent=True, t_max=400, n_max=4):
    keep = (t < t_max) & (m_used == m_true).flatten(1).all(1)
    if not keep.any():
        return None
    i = keep.nonzero()[:n_max, 0]
    a = sched.alphas_cumprod.to(t.device)[t[i]][:, None, None, None]
    z0 = (zt[i] - (1 - a).sqrt() * pred[i]) / a.sqrt()
    with torch.autocast(DEVICE, dtype=DTYPE):
        img = vae.decode(z0 / LATENT_SCALE).sample if latent else z0
        return seg_loss5(seg(img).float(), m_true[i])

def ldm_loss(unet, x, m=None, ablate=True, latent=True, anat=None):
    if latent:
        with torch.no_grad(), torch.autocast(DEVICE, dtype=DTYPE):
            z = vae.encode(x).latent_dist.mean.float() * LATENT_SCALE # frozen VAE, no grad
    else:
        z = x
    t = torch.randint(0, sched.config.num_train_timesteps, (len(z),), device=DEVICE)
    eps = torch.randn_like(z)
    zt = sched.add_noise(z, eps, t)
    inp, m_used = zt, None
    if m is not None:
        m_used = mask_ablate(m) if ablate else m
        inp = torch.cat([zt, cond_channels(m_used, hw=zt.shape[-2:])], 1)
    with torch.autocast(DEVICE, dtype=DTYPE):
        pred = unet(inp, t).sample
    loss = F.mse_loss(pred.float(), eps)
    if anat is not None and m is not None:
        seg, w = anat
        a = anat_term(seg, pred.float(), zt, t, m, m_used, latent=latent)
        if a is not None:
            loss = loss + adaptive_w(loss, a, unet.conv_out.weight) * w * a
    return loss


Validation on a fixed noise/timestep grid.

In [ ]:
@torch.no_grad()
def encode_latents(arr, batch=32):
    """Patches -> scaled latents."""
    out = []
    for i in range(0, len(arr), batch):
        x = torch.stack([torch.from_numpy(norm_img(a))[None] for a in arr[i:i + batch]]).to(DEVICE)
        with torch.autocast(DEVICE, dtype=DTYPE):
            out.append(vae.encode(x).latent_dist.mean.float() * LATENT_SCALE)
    return torch.cat(out)

Z_VAL = encode_latents(real_va)
set_seed()
T_VAL = torch.randint(0, sched.config.num_train_timesteps, (len(Z_VAL),), device=DEVICE)
E_VAL = torch.randn_like(Z_VAL)

@torch.no_grad()
def eval_ldm(unet, z, t, e, c=None, batch=16):
    """Fixed (t, eps), so the number is comparable across training steps."""
    unet.eval(); tot, n = 0.0, 0
    for i in range(0, len(z), batch):
        zt = sched.add_noise(z[i:i+batch], e[i:i+batch], t[i:i+batch])
        if c is not None:
            zt = torch.cat([zt, c[i:i+batch]], 1)
        with torch.autocast(DEVICE, dtype=DTYPE):
            pred = unet(zt, t[i:i+batch]).sample
        k = len(zt)
        tot += F.mse_loss(pred.float(), e[i:i+batch]).item() * k; n += k
    return tot / n

Sampling: DDIM with classifier-free guidance where a null condition exists.

In [ ]:
@torch.no_grad()
def sample_ldm(unet, n, steps=100, seed=SEED, c=None, w=1.0, latent=True):
    """Reverse process. c given -> classifier-free guidance:
       eps = eps(null) + w * (eps(c) - eps(null)), w=1 being plain conditional."""
    unet.eval()
    g = torch.Generator(DEVICE).manual_seed(seed)
    shape = (n, LATENT_CH, LATENT_H, LATENT_W) if latent else (n, 1, PATCH_H, PATCH_W)
    z = torch.randn(shape, device=DEVICE, generator=g)
    null = None
    if c is not None:
        null = torch.zeros_like(c); null[:, 0] = 1.0        # all background = condition 0
    sampler.set_timesteps(steps, device=DEVICE)
    for t in sampler.timesteps:
        with torch.autocast(DEVICE, dtype=DTYPE):
            if c is None:
                e = unet(z, t.expand(n)).sample.float()
            else:
                e_c = unet(torch.cat([z, c], 1), t.expand(n)).sample.float()
                e_u = unet(torch.cat([z, null], 1), t.expand(n)).sample.float()
                e = e_u + w * (e_c - e_u)
        z = sampler.step(e, t, z).prev_sample
    if latent:
        with torch.autocast(DEVICE, dtype=DTYPE):
            z = vae.decode(z / LATENT_SCALE).sample.float()
    return [denorm_img(a) for a in z[:, 0].cpu().numpy()]

Training: 10000 steps, effective batch 32, EMA.

In [ ]:
def train_ldm(unet, dl, val_fn, name="ldm", steps=10000, lr=1e-4, warmup=1000,
              val_every=500, ema_decay=0.9995, latent=True, accum=1, anat=None):
    """One function for all three models."""
    torch.cuda.reset_peak_memory_stats()
    ckpt, meta = CKPT / f"{name}.pt", CKPT / f"{name}_meta.json"
    opt = torch.optim.AdamW(unet.parameters(), lr)
    ema = EMAModel(unet.parameters(), decay=ema_decay)

    def lr_at(i):
        if i < warmup:
            return i / warmup
        return 0.5 * (1 + math.cos(math.pi * (i - warmup) / max(1, steps - warmup)))
    lr_sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)

    it_dl = iter(dl)
    def next_batch():
        nonlocal it_dl
        b = next(it_dl, None)
        if b is None:
            it_dl = iter(dl); b = next(it_dl)
        return b

    hist, it, best = dict(step=[], train=[], val_step=[], val=[]), 0, float("inf")
    pbar = tqdm(total=steps, desc=name)
    while it < steps:
        unet.train()
        for _ in range(accum):
            b = next_batch()
            x, m = b if isinstance(b, (list, tuple)) else (b, None)
            loss = ldm_loss(unet, x.to(DEVICE, non_blocking=True),
                            None if m is None else m.to(DEVICE, non_blocking=True),
                            latent=latent, anat=anat)
            (loss / accum).backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True); lr_sched.step()
        ema.step(unet.parameters())
        it += 1; pbar.update(1)
        hist["step"].append(it); hist["train"].append(loss.item())

        if it % val_every == 0 or it == steps:
            ema.store(unet.parameters()); ema.copy_to(unet.parameters())
            v = val_fn(unet)
            if v < best:
                best = v
                torch.save(unet.state_dict(), ckpt)
            ema.restore(unet.parameters())
            hist["val_step"].append(it); hist["val"].append(v)
            pbar.set_postfix(train=f"{loss.item():.4f}", val_ema=f"{v:.4f}", best=f"{best:.4f}")
    pbar.close()

    meta.write_text(json.dumps({"hist": hist, "steps": steps, "best_val": best}))
    print(f"best val MSE (EMA) {best:.4f} | "
          f"peak VRAM {torch.cuda.max_memory_allocated()/2**20:.0f} MiB")
    return hist

set_seed()
unet = make_unet().to(DEVICE)

if (CKPT / "ldm.pt").exists():
    hist_ldm = json.loads(LDM_META.read_text()).get("hist")
    print("LDM: cached checkpoint, training skipped (delete cache/ckpt/ldm.pt to retrain)")
else:
    hist_ldm = train_ldm(unet, ldm_dl, lambda net: eval_ldm(net, Z_VAL, T_VAL, E_VAL))

unet.load_state_dict(torch.load(CKPT / "ldm.pt"))
unet.eval()

## S_build

In [ ]:
@torch.no_grad()
def eval_seg(net, dl, thr=0.5):
    """Mean per-image Dice. Per-image, not aggregated."""
    net.eval(); tot, n = 0.0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE).float()[:, None]
        with torch.autocast(DEVICE, dtype=DTYPE):
            p = (torch.sigmoid(net(x).float()) > thr).float()
        d = (2 * (p * y).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
        tot += d.sum().item(); n += len(d)
    return tot / n

set_seed()
s_build = SegUNet(out_ch=1).to(DEVICE)

if (CKPT / "s_build.pt").exists():
    print("S_build: cached checkpoint, training skipped (delete cache/ckpt/s_build.pt to retrain)")
else:
    torch.cuda.reset_peak_memory_stats()
    steps = 20 * len(build_dl)                      # ~20 epochs over the 8977 aux patches
    opt = torch.optim.AdamW(s_build.parameters(), 1e-3)
    lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    best, it = 0.0, 0
    pbar = tqdm(total=steps, desc="s_build")
    while it < steps:
        for x, y in build_dl:
            if it >= steps:
                break
            s_build.train()
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE).float()[:, None]
            with torch.autocast(DEVICE, dtype=DTYPE):
                loss = seg_loss(s_build(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(s_build.parameters(), 1.0)
            opt.step(); opt.zero_grad(set_to_none=True); lr_sched.step()
            it += 1; pbar.update(1)
            if it % 500 == 0 or it == steps:
                d = eval_seg(s_build, build_dlv)
                if d > best:
                    best = d; torch.save(s_build.state_dict(), CKPT / "s_build.pt")
                pbar.set_postfix(loss=f"{loss.item():.4f}", val_dice=f"{d:.4f}", best=f"{best:.4f}")
    pbar.close()
    print(f"best val Dice {best:.4f} | peak VRAM {torch.cuda.max_memory_allocated()/2**20:.0f} MiB")

s_build.load_state_dict(torch.load(CKPT / "s_build.pt"))
s_build.eval()
# printed on every run, cached or not: the README quotes this number
print(f"S_build val Dice {eval_seg(s_build, build_dlv):.4f}")
for p in s_build.parameters():
    p.requires_grad_(False)

In [ ]:
TOOTH_NPZ = CACHE / "tooth_masks.npz"

if TOOTH_NPZ.exists():
    TOOTH = np.load(TOOTH_NPZ)["m"]
else:
    out = []
    for i in range(0, len(PATCHES), 32):
        x = torch.stack([torch.from_numpy(norm_img(p["patch"]))[None]
                         for p in PATCHES[i:i + 32]]).to(DEVICE)
        with torch.no_grad(), torch.autocast(DEVICE, dtype=DTYPE):
            out.append((torch.sigmoid(s_build(x).float()) > 0.5)[:, 0].cpu().numpy())
    TOOTH = np.concatenate(out).astype(np.uint8)
    np.savez_compressed(TOOTH_NPZ, m=TOOTH)

print(f"silhouettes {TOOTH.shape} | fill fraction mean {TOOTH.mean():.3f} (aux GT 0.396)")

Masks and sanity check: severity is recoverable from a mask.

In [ ]:
MASKS = np.zeros((len(PATCHES), PATCH_H, PATCH_W), np.uint8)
rows, dropped = [], []
for i, (p, t) in enumerate(zip(PATCHES, TOOTH)):
    sides = [v for v in (p["pbl_m"], p["pbl_d"]) if v is not None]
    m = build_mask(t, p["kp"])
    # an anatomically impossible side means its BL keypoint is unreliable: drop the tooth
    if m is None or not sides or max(sides) > PBL_MAX:
        dropped.append(i); continue
    MASKS[i] = m
    r = realized_pbl(m)
    if r is None:
        dropped.append(i); continue
    # the mask crest line interpolates between the two sides, so the target is their mean,
    # not the worst-side PBL used for staging
    rows.append((i, r, float(np.mean(sides))))

idx  = np.array([r[0] for r in rows])
got  = np.array([r[1] for r in rows])
want = np.array([r[2] for r in rows])
err  = np.abs(got - want)

print(f"masks {len(rows)} built, {len(dropped)} dropped (missing or invalid landmarks)")
print(f"realized vs keypoint PBL: MAE {err.mean():.4f}  median {np.median(err):.4f}  "
      f"p90 {np.percentile(err, 90):.4f}  max {err.max():.4f}")
print(f"  bias {np.mean(got - want):+.4f}   frac > 0.02: {(err > 0.02).mean():.2f}")
print(f"  stage agreement: {np.mean([stage_of(a) == stage_of(b) for a, b in zip(got, want)]):.3f}")
cls = np.array([PATCHES[i]["cls"] for i in idx])
for c in TOOTH_CLASSES:
    s = cls == c
    print(f"  {BOX_CLASSES[c]:12s} n={s.sum():3d}  MAE {err[s].mean():.4f}")

OK = err.mean() < 0.02
print("\nSANITY CHECK:", "PASS - PBL is recoverable from the mask, so it is commandable" if OK else
      "FAIL - the mask does not encode PBL")
assert OK, "mask construction does not reproduce the keypoint PBL"

## S_eval

In [ ]:
# S_eval is trained on REAL patches with the constructed masks, S_build on AUX patches
# with polygon truth. One builds the conditioning, the other
# judges it, and using one net for both would make every faithfulness number circular.
drop = set(dropped)
seg_tr = [i for i, p in enumerate(PATCHES) if p["split"] == "train" and i not in drop]
seg_va = [i for i, p in enumerate(PATCHES) if p["split"] == "val"   and i not in drop]
seg_te = [i for i, p in enumerate(PATCHES) if p["split"] == "test"  and i not in drop]

set_seed()
eval_dl = torch.utils.data.DataLoader(
    PatchDataset(np.stack([PATCHES[i]["patch"] for i in seg_tr]), MASKS[seg_tr],
                 train=True, affine=True),          # no vflip: real patches are crown-up
    16, shuffle=True, drop_last=True)
EVAL_XV = np.stack([PATCHES[i]["patch"] for i in seg_va])
EVAL_YV = MASKS[seg_va]
print(f"S_eval train {len(seg_tr)} | val {len(seg_va)}")

@torch.no_grad()
def predict_masks(net, arr, batch=16, tta=True):
    """Averaged with the horizontal flip: the mask classes are not side-specific, so the
       flip is a free second opinion."""
    net.eval(); out = []
    for i in range(0, len(arr), batch):
        x = torch.stack([torch.from_numpy(norm_img(a))[None] for a in arr[i:i + batch]]).to(DEVICE)
        with torch.autocast(DEVICE, dtype=DTYPE):
            p = net(x).float().softmax(1)
            if tta:
                p = p + torch.flip(net(torch.flip(x, [3])).float().softmax(1), [3])
        out.append(p.argmax(1).cpu().numpy().astype(np.uint8))
    return np.concatenate(out)

def train_seg5(net, name, select=2, steps=4000):
    """select: index of the class whose val Dice picks the checkpoint, or None for the mean
       over classes. Written once because S_eval2 uses it with a different criterion."""
    torch.cuda.reset_peak_memory_stats()
    opt = torch.optim.AdamW(net.parameters(), 1e-3)
    lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    best, it = 0.0, 0
    pbar = tqdm(total=steps, desc=name)
    while it < steps:
        for x, y in eval_dl:
            if it >= steps:
                break
            net.train()
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            with torch.autocast(DEVICE, dtype=DTYPE):
                loss = seg_loss5(net(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step(); opt.zero_grad(set_to_none=True); lr_sched.step()
            it += 1; pbar.update(1)
            if it % 250 == 0 or it == steps:
                d = np.nanmean([dice_per_class(a, b) for a, b in
                                zip(predict_masks(net, EVAL_XV), EVAL_YV)], 0)
                v = d.mean() if select is None else d[select]
                if v > best:
                    best = v; torch.save(net.state_dict(), CKPT / f"{name}.pt")
                pbar.set_postfix(loss=f"{loss.item():.4f}", mean=f"{d.mean():.3f}",
                                 c2=f"{d[2]:.3f}", best=f"{best:.3f}")
    pbar.close()
    print(f"{name}: best val Dice {best:.4f} | "
          f"peak VRAM {torch.cuda.max_memory_allocated()/2**20:.0f} MiB")

set_seed()
s_eval = SegUNet(out_ch=N_MASK_CLASSES).to(DEVICE)

if (CKPT / "s_eval.pt").exists():
    print("S_eval: cached checkpoint, training skipped (delete cache/ckpt/s_eval.pt to retrain)")
else:
    train_seg5(s_eval, "s_eval", select=2)

s_eval.load_state_dict(torch.load(CKPT / "s_eval.pt"))
s_eval.eval()
for p in s_eval.parameters():
    p.requires_grad_(False)


## S_eval2

Same architecture, but different seed, batch order and selection criterion. Needed because S_eval is used in the anatomical loss.

In [ ]:
set_seed(SEED + 1)
s_eval2 = SegUNet(out_ch=N_MASK_CLASSES).to(DEVICE)

if (CKPT / "s_eval2.pt").exists():
    print("S_eval2: cached checkpoint, training skipped (delete cache/ckpt/s_eval2.pt to retrain)")
else:
    train_seg5(s_eval2, "s_eval2", select=None)

s_eval2.load_state_dict(torch.load(CKPT / "s_eval2.pt"))
s_eval2.eval()
for p in s_eval2.parameters():
    p.requires_grad_(False)

p2 = predict_masks(s_eval2, np.stack([PATCHES[i]["patch"] for i in seg_va]))
w2 = np.array([realized_pbl(MASKS[i]) for i in seg_va])
g2 = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in p2])
k2 = ~np.isnan(g2) & ~np.isnan(w2)
d2 = np.nanmean([dice_per_class(a, b) for a, b in zip(p2, MASKS[seg_va])], 0)
print(f"S_eval2 val Dice per class {np.round(d2, 3)} | mean {d2.mean():.3f}")
print(f"S_eval2 measurement floor: PBL MAE {np.abs(g2[k2] - w2[k2]).mean():.4f} "
      f"(S_eval 0.0436)")


## Conditioned LDM

In [ ]:
set_seed()
cond_dl = torch.utils.data.DataLoader(
    PatchDataset(np.stack([PATCHES[i]["patch"] for i in seg_tr]), MASKS[seg_tr],
                 train=True, affine=True), 32, shuffle=True, drop_last=True)

ZC_VAL = encode_latents(np.stack([PATCHES[i]["patch"] for i in seg_va]))
C_VAL  = cond_channels(torch.from_numpy(MASKS[seg_va]).to(DEVICE))
set_seed()
TC_VAL = torch.randint(0, sched.config.num_train_timesteps, (len(ZC_VAL),), device=DEVICE)
EC_VAL = torch.randn_like(ZC_VAL)
print(f"conditioned LDM: train {len(seg_tr)} | val {len(ZC_VAL)} | condition {tuple(C_VAL.shape[1:])}")

Sanity check

In [ ]:
def check(steps=1500, n=16, sample_steps=25):
    """Overfit n (patch, mask) pairs with ablation off, then sample from those same masks
       and ask whether the commanded PBL comes back out, measured through S_eval."""
    # spread across severity, not the first n by index: PATCHES is ordered by radiograph,
    # so seg_tr[:n] would be adjacent teeth of a handful of images, all Healthy/Mild
    pbl_tr = [np.mean([v for v in (PATCHES[i]["pbl_m"], PATCHES[i]["pbl_d"]) if v is not None])
              for i in seg_tr]
    o = np.argsort(pbl_tr)
    sel = [seg_tr[o[k]] for k in np.linspace(0, len(o) - 1, n).astype(int)]
    dup = sum(float((MASKS[a] == MASKS[b]).mean()) > 0.90
              for ai, a in enumerate(sel) for b in sel[ai + 1:])

    set_seed()
    net = make_unet(mask_ch=N_MASK_CLASSES).to(DEVICE)
    x = torch.stack([torch.from_numpy(norm_img(PATCHES[i]["patch"]))[None] for i in sel]).to(DEVICE)
    m = torch.from_numpy(MASKS[sel]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), 1e-4)
    for _ in tqdm(range(steps), desc="check"):
        net.train()
        loss = ldm_loss(net, x, m=m, ablate=False)
        loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True)

    imgs = sample_ldm(net, n, steps=sample_steps, c=cond_channels(m))
    want = np.array([realized_pbl(MASKS[i]) for i in sel])
    got  = np.array([realized_pbl(p) if realized_pbl(p) is not None else np.nan
                     for p in predict_masks(s_eval, np.stack(imgs))])
    ok = ~np.isnan(got)
    corr = float(np.corrcoef(got[ok], want[ok])[0, 1])
    mae  = float(np.abs(got[ok] - want[ok]).mean())

    ref = np.stack([PATCHES[i]["patch"] for i in sel]).astype(np.float32)
    d = np.array([[np.abs(ref[j] - s.astype(np.float32)).mean() for j in range(n)] for s in imgs])
    hit = (d.argmin(1) == np.arange(n)).mean()

    print(f"final loss {loss.item():.4f} | commanded PBL {want.min():.2f}-{want.max():.2f}, "
          f"{dup}/{n*(n-1)//2} confusable mask pairs")
    print(f"  commanded vs realized PBL: corr {corr:+.3f}  MAE {mae:.4f}  "
          f"(S_eval floor {0.044:.3f})   n={ok.sum()}/{n}")
    print(f"  identity: sample from mask i nearest to patch i {int(hit*n)}/{n}  "
          f"(chance {1/n:.2f}) -- reported, not gated")
    print("Sanity check:", "PASS" if corr > 0.5 else "FAIL")
    assert corr > 0.5, "the commanded PBL does not come back out: mask and latent are misaligned"
    return imgs, sel

G5, G5_SEL = check()
show_grid([PATCHES[i]["patch"] for i in G5_SEL[::2]] + G5[::2],
          ["train"] * 8 + ["from its mask"] * 8, ncols=8, size=2.0, save="b4_gate5.png")
torch.cuda.empty_cache()

Training

In [ ]:
set_seed()
unet_c = make_unet(mask_ch=N_MASK_CLASSES).to(DEVICE)
print(f"{sum(p.numel() for p in unet_c.parameters())/1e6:.1f}M params | "
      f"{LATENT_CH}+{N_MASK_CLASSES}ch -> {LATENT_CH}ch")

if (CKPT / "ldm_cond.pt").exists():
    hist_cond = json.loads((CKPT / "ldm_cond_meta.json").read_text()).get("hist")
    print("conditioned LDM: cached checkpoint, training skipped "
          "(delete cache/ckpt/ldm_cond.pt to retrain)")
else:
    hist_cond = train_ldm(unet_c, cond_dl,
                          lambda net: eval_ldm(net, ZC_VAL, TC_VAL, EC_VAL, C_VAL),
                          name="ldm_cond")

unet_c.load_state_dict(torch.load(CKPT / "ldm_cond.pt"))
unet_c.eval()

## DDPM

Same conditions as the LDM (data, jitter, masks, optimizer steps, effective batch).

In [ ]:
set_seed()
px_dl = torch.utils.data.DataLoader(
    PatchDataset(np.stack([PATCHES[i]["patch"] for i in seg_tr]), MASKS[seg_tr],
                 train=True, affine=True), 8, shuffle=True, drop_last=True)

ZP_VAL = torch.stack([torch.from_numpy(norm_img(PATCHES[i]["patch"]))[None]
                      for i in seg_va]).to(DEVICE)
CP_VAL = cond_channels(torch.from_numpy(MASKS[seg_va]).to(DEVICE), hw=(PATCH_H, PATCH_W))
set_seed()
TP_VAL = torch.randint(0, sched.config.num_train_timesteps, (len(ZP_VAL),), device=DEVICE)
EP_VAL = torch.randn_like(ZP_VAL)

set_seed()
unet_px = make_unet_px().to(DEVICE)

if (CKPT / "ddpm_px.pt").exists():
    hist_px = json.loads((CKPT / "ddpm_px_meta.json").read_text()).get("hist")
    print("pixel DDPM: cached checkpoint, training skipped "
          "(delete cache/ckpt/ddpm_px.pt to retrain)")
else:
    # needed as to not exhaust vRAM
    parked = [vae, unet, unet_c, s_build, s_eval, lpips_fn]
    for mdl in parked:
        mdl.cpu()
    torch.cuda.empty_cache()
    unet_px.enable_gradient_checkpointing()
    free, total = torch.cuda.mem_get_info()
    print(f"free VRAM {free/2**20:.0f} / {total/2**20:.0f} MiB")
    assert free / 2**20 > 5500, "restart the kernel"
    hist_px = train_ldm(unet_px, px_dl,
                        lambda net: eval_ldm(net, ZP_VAL, TP_VAL, EP_VAL, CP_VAL, batch=4),
                        name="ddpm_px", latent=False, accum=4)
    for mdl in parked:
        mdl.to(DEVICE)

unet_px.load_state_dict(torch.load(CKPT / "ddpm_px.pt"))
unet_px.eval()

# Synthetic Banks

400 patches per model, commanded 100% Advanced and 60% Severe, guided and unguided.

In [ ]:
N_BANK = 400
BANK_NPZ = CACHE / "banks.npz"

# Severity is commanded, not sampled from the data: 60% Severe / 40% Moderate, on TRAIN
# silhouettes drawn with replacement. Only the crest moves.
rng = np.random.default_rng(SEED)
# Reuse real teeth and change the crest.
bank_src = rng.choice(seg_tr, N_BANK, replace=True)
bank_tgt = np.where(rng.random(N_BANK) < 0.6, rng.uniform(SEVERE_MIN, 0.85, N_BANK),
                                              rng.uniform(ADVANCED_MIN, SEVERE_MIN, N_BANK))
BANK_M = np.stack([retarget_mask(MASKS[i], t) for i, t in zip(bank_src, bank_tgt)])

def gen_bank(net, masks, latent, guided, batch, seed=SEED):
    """guided=False feeds the all-background map, which is the null condition MAT taught the
       model via p_null. It is the H2 control: same generator, same count, no anatomy."""
    hw = (LATENT_H, LATENT_W) if latent else (PATCH_H, PATCH_W)
    out = []
    for k in tqdm(range(0, len(masks), batch), desc=f"{'lat' if latent else 'px'}/"
                                                    f"{'guided' if guided else 'unguided'}"):
        mm = masks[k:k + batch]
        c = cond_channels(torch.from_numpy(mm if guided else np.zeros_like(mm)).to(DEVICE), hw=hw)
        out += sample_ldm(net, len(c), steps=100, seed=seed + k, c=c, w=1.0, latent=latent)
    return np.stack(out)

if BANK_NPZ.exists():
    d = np.load(BANK_NPZ)
    BANKS = {k: d[k] for k in d.files if k != "tgt"}
    print("banks: cached (delete cache/banks.npz to regenerate)")
else:
    BANKS = {"lat_g":  gen_bank(unet_c,  BANK_M, True,  True,  25),
             "lat_u":  gen_bank(unet_c,  BANK_M, True,  False, 25),
             "px_g":   gen_bank(unet_px, BANK_M, False, True,  12),
             "px_u":   gen_bank(unet_px, BANK_M, False, False, 12)}
    np.savez_compressed(BANK_NPZ, tgt=bank_tgt, **BANKS)

# Measure realized PBL so the images can be relabeled after.
BANK_PBL = {k: np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan
                         for m in predict_masks(s_eval, v)]) for k, v in BANKS.items()}

print(f"commanded PBL: {bank_tgt.min():.2f}-{bank_tgt.max():.2f}, "
      f"{(bank_tgt >= SEVERE_MIN).mean()*100:.0f}% Severe")
for k, v in BANKS.items():
    p = BANK_PBL[k][~np.isnan(BANK_PBL[k])]
    print(f"  {k:6s} {str(v.shape):14s} realized {p.mean():.3f} +-{p.std():.3f}  "
          f"{(p >= ADVANCED_MIN).mean()*100:3.0f}% Advanced")

## Classifier

11 arms × 5 seeds, class-balanced sampling, identical schedule, checkpoint selected on a real-only validation set in every arm. Seeds are paired across arms, so the per-seed difference cancels the seed effect the arms share.

In [ ]:
CLF_JSON = CACHE / "downstream.json"
CLF_SEEDS, CLF_EPOCHS = 5, 30

real_x = np.stack([PATCHES[i]["patch"] for i in seg_tr])
real_y = np.array([PATCHES[i]["advanced"] for i in seg_tr], int)
val_x  = np.stack([PATCHES[i]["patch"] for i in seg_va])
val_y  = np.array([PATCHES[i]["advanced"] for i in seg_va], int)
test_x = np.stack([PATCHES[i]["patch"] for i in seg_te])
test_y = np.array([PATCHES[i]["advanced"] for i in seg_te], int)

# Three labelling regimes over the same four banks, because "the guidance does not help"
# and "the labels are wrong" are two different findings and the intent labels confound them.
#   B/D  by INTENT   -> every bank was commanded Advanced (what a user of the engine gets)
#   E/F  by MEASURE  -> label = S_eval reads PBL >= 0.33 (removes the label noise)
#   G    FILTERED    -> keep only the patches that are genuinely Advanced (adds rare cases)
ARMS = {"A  real only": (real_x, real_y)}
ARM_INFO = {"A  real only": (0, np.nan, np.nan)} # n synthetic, labelled Adv, truly Adv

def add_arm(lbl, k, keep, lab):
    """keep: bool mask over bank k. lab: labels for the kept patches."""
    ARMS[lbl] = (np.concatenate([real_x, BANKS[k][keep]]),
                 np.concatenate([real_y, lab]))
    p = BANK_PBL[k][keep]
    ARM_INFO[lbl] = (int(keep.sum()), float(lab.mean()),
                     float((p[~np.isnan(p)] >= ADVANCED_MIN).mean()))

BANK_LBL = (("lat_g", "latent guided"), ("px_g", "pixel guided"),
            ("lat_u", "latent unguided"), ("px_u", "pixel unguided"))
for (k, what), tag in zip(BANK_LBL, ("B_lat ", "B_px  ", "D_lat ", "D_px  ")):
    all_of = np.ones(len(BANKS[k]), bool)
    add_arm(f"{tag} + {what}", k, all_of, np.ones(len(BANKS[k]), int))
for (k, what), tag in zip(BANK_LBL, ("E_lat ", "E_px  ", "F_lat ", "F_px  ")):
    ok = ~np.isnan(BANK_PBL[k])
    add_arm(f"{tag} {what}, measured", k, ok, (BANK_PBL[k][ok] >= ADVANCED_MIN).astype(int))
for (k, what), tag in zip(BANK_LBL[:2], ("G_lat ", "G_px  ")):
    ok = ~np.isnan(BANK_PBL[k]) & (BANK_PBL[k] >= ADVANCED_MIN)
    add_arm(f"{tag} {what}, Advanced only", k, ok, np.ones(int(ok.sum()), int))

@torch.no_grad()
def clf_prob(net, x, bs=32):
    net.eval(); out = []
    for i in range(0, len(x), bs):
        xb = torch.stack([torch.from_numpy(norm_img(a))[None] for a in x[i:i + bs]]).to(DEVICE)
        with torch.autocast(DEVICE, dtype=DTYPE):
            out.append(net(xb).float().softmax(1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def train_clf(x, y, seed):
    """Class-balanced sampling, identical schedule for every arm, checkpoint selected on VAL
       macro-F1. VAL is real only in every arm: an arm must not be scored on its own
       synthetic distribution."""
    set_seed(seed)
    net = make_clf().to(DEVICE)
    w = (1.0 / np.bincount(y, minlength=2))[y]
    dl = torch.utils.data.DataLoader(
        ClsDataset(x, y, train=True), 32, drop_last=True,
        sampler=torch.utils.data.WeightedRandomSampler(torch.from_numpy(w), len(y), True))
    opt = torch.optim.AdamW(net.parameters(), 1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CLF_EPOCHS)
    best, state = -1.0, None
    for _ in range(CLF_EPOCHS):
        net.train()
        for xb, yb in dl:
            with torch.autocast(DEVICE, dtype=DTYPE):
                loss = F.cross_entropy(net(xb.to(DEVICE)), yb.to(DEVICE))
            loss.backward()
            opt.step(); opt.zero_grad(set_to_none=True)
        sch.step()
        f = macro_f1(val_y, clf_prob(net, val_x))
        if f > best:
            best, state = f, {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(state)
    return net

CLF = json.loads(CLF_JSON.read_text()) if CLF_JSON.exists() else {}
for name, (x, y) in ARMS.items():
    if name in CLF:
        continue
    rows = []
    for s in tqdm(range(CLF_SEEDS), desc=name.split()[0]):
        p = clf_prob(train_clf(x, y, SEED + s), test_x)
        rows.append(dict(mAP=0.5 * (average_precision(test_y, p) +
                                    average_precision(1 - test_y, -p)),
                         f1=macro_f1(test_y, p),
                         ap_adv=average_precision(test_y, p),
                         tp=int(((p >= 0.5) & (test_y == 1)).sum()),
                         fp=int(((p >= 0.5) & (test_y == 0)).sum())))
    CLF[name] = rows
    CLF_JSON.write_text(json.dumps(CLF, indent=1))
    torch.cuda.empty_cache()
print(f"{len(CLF)} arms x {CLF_SEEDS} seeds, TEST n={len(test_y)} ({test_y.sum()} Advanced)")

## Anatomical LDM


LDM retrained with the anatomical term.

In [ ]:
# The anatomical term reaches the UNet through the VAE decoder, so the decoder needs to
# pass gradients but must not collect them.
for p in vae.parameters():
    p.requires_grad_(False)

set_seed()
unet_a = make_unet(mask_ch=N_MASK_CLASSES).to(DEVICE)

if (CKPT / "ldm_anat.pt").exists():
    hist_anat = json.loads((CKPT / "ldm_anat_meta.json").read_text()).get("hist")
    print("anatomical LDM: cached checkpoint, training skipped "
          "(delete cache/ckpt/ldm_anat.pt to retrain)")
else:
    parked = [unet, unet_c, unet_px, s_build, s_eval2, lpips_fn, disc]
    for mdl in parked:
        mdl.cpu()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"free VRAM {free/2**20:.0f} / {total/2**20:.0f} MiB")
    try:
        assert free / 2**20 > 5000, "something else is holding the GPU"
        hist_anat = train_ldm(unet_a, cond_dl,
                              lambda net: eval_ldm(net, ZC_VAL, TC_VAL, EC_VAL, C_VAL),
                              name="ldm_anat", anat=(s_eval, 0.25))
    finally:
        for mdl in parked:
            mdl.to(DEVICE)

unet_a.load_state_dict(torch.load(CKPT / "ldm_anat.pt"))
unet_a.eval()


## Anatomical DDPM

Compare the previous results with this to understand whether the addition of the anatomical term improves the performance also in the pixel space.

In [ ]:
set_seed()
unet_px_a = make_unet_px().to(DEVICE)

if (CKPT / "ddpm_px_anat.pt").exists():
    hist_px_a = json.loads((CKPT / "ddpm_px_anat_meta.json").read_text()).get("hist")
    print("pixel DDPM + anatomical: cached checkpoint, training skipped "
          "(delete cache/ckpt/ddpm_px_anat.pt to retrain)")
else:
    parked = [vae, unet, unet_c, unet_a, unet_px, s_build, s_eval2, lpips_fn, disc]
    for mdl in parked:
        mdl.cpu()
    torch.cuda.empty_cache()
    PX_BS, PX_ACC = 2, 16
    unet_px_a.enable_gradient_checkpointing()
    set_seed()
    px_a_dl = torch.utils.data.DataLoader(
        PatchDataset(np.stack([PATCHES[i]["patch"] for i in seg_tr]), MASKS[seg_tr],
                     train=True, affine=True), PX_BS, shuffle=True, drop_last=True)
    free, total = torch.cuda.mem_get_info()
    print(f"free VRAM {free/2**20:.0f} / {total/2**20:.0f} MiB")
    try:
        assert free / 2**20 > 4000, "something else is using the GPU"
        hist_px_a = train_ldm(unet_px_a, px_a_dl,
                              lambda net: eval_ldm(net, ZP_VAL, TP_VAL, EP_VAL, CP_VAL, batch=4),
                              name="ddpm_px_anat", latent=False, accum=PX_ACC,
                              anat=(s_eval, 0.25))
    finally:
        for mdl in parked:
            mdl.to(DEVICE)

unet_px_a.load_state_dict(torch.load(CKPT / "ddpm_px_anat.pt"))
unet_px_a.eval()


# Test

Every metric is reported next to what real images score through the same instrument: `S_eval`'s own error on real patches is the floor, real-image Dice is the ceiling, `w=0` is the chance anchor. Absolute numbers on generated images carry the disagreement between the two evaluators; the comparisons between models are the load-bearing part, and both judges agree on the sign and size of every one of them.

## VAE

The crest must not be damaged more than the rest of the patch, and the patch must not be blurred flat.

In [ ]:
# calibrate the crest_ratio threshold against known-answer perturbations
def _blur(a, k=5): return cv2.GaussianBlur(a, (k, k), 0)

rng = np.random.default_rng(SEED)
te  = [p for p in PATCHES if p["split"] == "test"]
for name, fn in (("identity       ", lambda a, kp: a),
                 ("uniform blur   ", lambda a, kp: _blur(a)),
                 ("uniform noise  ", lambda a, kp: np.clip(a + rng.normal(0, 6, a.shape), 0, 255).astype(np.uint8)),
                 ("crest-only blur", lambda a, kp: np.where(crest_band(kp), _blur(a, 9), a).astype(np.uint8))):
    r = [x for x in (crest_ratio(p["patch"], fn(p["patch"], p["kp"]), p["kp"]) for p in te) if x is not None]
    print(f"  {name}: median {np.median(r):.3f}  p90 {np.percentile(r, 90):.3f}")

In [ ]:
ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(DEVICE)

vae.eval(); RECS, ROWS = [], []
with torch.no_grad():
    for i in range(0, len(te), 16):
        chunk = te[i:i + 16]
        x = torch.stack([torch.from_numpy(norm_img(p["patch"]))[None] for p in chunk]).to(DEVICE)
        with torch.autocast(DEVICE, dtype=DTYPE):
            r = vae(x).sample.float()
        mse = ((r - x) ** 2).mean(dim=(1, 2, 3))
        for j, p in enumerate(chunk):
            rec = denorm_img(r[j, 0].cpu().numpy())
            RECS.append(rec)
            ROWS.append(dict(psnr=float(10 * torch.log10(4.0 / mse[j])),
                             ssim=float(ssim_fn(r[j:j+1], x[j:j+1])),
                             lpips=float(perceptual(r[j:j+1], x[j:j+1])),
                             ratio=crest_ratio(p["patch"], rec, p["kp"]),
                             hf=hf_energy(rec) / (hf_energy(p["patch"]) + 1e-8)))

print(f"TEST n={len(ROWS)}")
for k in ("psnr", "ssim", "lpips"):
    v = np.array([r[k] for r in ROWS])
    q = 95 if k == "lpips" else 5   # LPIPS is lower-is-better: the bad tail is on top
    print(f"  {k:5s} mean {v.mean():.4f}  p{q} {np.percentile(v, q):.4f}")

ratios = np.array([r["ratio"] for r in ROWS if r["ratio"] is not None])
hfs    = np.array([r["hf"] for r in ROWS])
print(f"  crest/global gradient-MAE: median {np.median(ratios):.3f}  "
      f"p90 {np.percentile(ratios, 90):.3f}  frac>1.25 {(ratios > 1.25).mean():.2f}")
print(f"  HF retained (sigma=1.5): mean {hfs.mean():.3f}  p5 {np.percentile(hfs, 5):.3f}")

g2a, g2b = np.median(ratios) < 1.25, hfs.mean() > 0.75
print(f"\nSanity check (crest not singled out) : {'PASS' if g2a else 'FAIL'}")
print(f"Sanity check (patch not blurred flat): {'PASS' if g2b else 'FAIL'}   target 0.75 set before the run")

In [ ]:
orig = [p["patch"] for p in te]
rfid, rkid, rkid_sd = fid_kid(orig, RECS)
print(f"rFID {rfid:.1f} | rKID {rkid:.4f} +- {rkid_sd:.4f}   (n={len(te)})")
print("  reference point for the decoder alone, not a floor on generation FID (Xu et al. 2026)")

# per scale: trabecular bone lives at 1-3 px, so the first two rows are the ones that matter
print("high-frequency energy retained (1.00 = perfect):")
for s in (0.8, 1.5, 3.0, 6.0):
    keep = np.mean([hf_energy(r, s) / (hf_energy(o, s) + 1e-8) for o, r in zip(orig, RECS)])
    print(f"  finer than {s:>4.1f} px: {keep:.3f}")

In [ ]:
order = np.argsort([-p["pbl"] for p in te])[:8]
imgs, titles = [], []
for i in order:
    o, r = te[i]["patch"], RECS[i]
    d = np.abs(o.astype(np.int16) - r.astype(np.int16)).astype(np.uint8)
    band = crest_band(te[i]["kp"])
    trio = np.hstack([to_canvas(o), to_canvas(r), to_canvas(np.clip(d * 4, 0, 255))])
    if band is not None:
        (x1, y1), (x2, y2) = te[i]["kp"][KP["BL_M"], :2], te[i]["kp"][KP["BL_D"], :2]
        for off in (-16, 16):              # the actual +-half strip, not its row extent
            cv2.line(trio, (2 * PATCH_W, int(y1 + off)), (3 * PATCH_W - 1, int(y2 + off)),
                     (0, 255, 255), 1)

    imgs.append(trio); titles.append(f'PBL {te[i]["pbl"]:.2f}  ratio {ROWS[i]["ratio"]:.2f}')
show_grid(imgs, titles, ncols=4, size=3.6, save="a1_vae_probe.png")


## LDM

In [ ]:
GEN = []
for k in range(0, 200, 25):
    GEN += sample_ldm(unet, 25, steps=100, seed=SEED + k)   # seed varies or the bank repeats
REAL_TE = [p["patch"] for p in te]
print(f"generated {len(GEN)} | real TEST {len(REAL_TE)}")

FID, KID, diversity, profile probe (samples that fall below the 5th percentile of real images on vertical-profile correlation, showing crown and root in the wrong order, crests that do not exist, etc.)

In [ ]:
fid_g, kid_g, kid_sd = fid_kid(REAL_TE, GEN)
div_g, div_r = lpips_diversity(GEN), lpips_diversity(REAL_TE)

print(f"LDM   FID {fid_g:6.1f} | KID {kid_g:.4f} +- {kid_sd:.4f} | LPIPS diversity {div_g:.3f}")
print(f"VAE   FID {rfid:6.1f} | KID {rkid:.4f} +- {rkid_sd:.4f}      <- decoder-only reference")
print(f"real                                          LPIPS diversity {div_r:.3f}")

In [ ]:
PROF_REAL = np.stack([row_profile(a) for a in REAL_TE]).mean(0)

def prof_corr(a):
    """Correlation with the mean real vertical profile. Low = the crown-up anatomy is
       broken, which no global metric like FID would single out."""
    return float(np.corrcoef(row_profile(a), PROF_REAL)[0, 1])

cr, cg = np.array([prof_corr(a) for a in REAL_TE]), np.array([prof_corr(a) for a in GEN])
hr, hg = np.array([hf_energy(a) for a in REAL_TE]), np.array([hf_energy(a) for a in GEN])

print(f"{'':22s} {'real':>16} {'generated':>16}")
print(f"{'vertical profile':22s} {cr.mean():7.3f} +-{cr.std():5.3f} {cg.mean():7.3f} +-{cg.std():5.3f}")
print(f"{'high-freq energy':22s} {hr.mean():7.2f} +-{hr.std():5.2f} {hg.mean():7.2f} +-{hg.std():5.2f}")
print(f"\nbroken profile (corr < {np.percentile(cr, 5):.2f}, the 5th percentile of real): "
      f"{(cg < np.percentile(cr, 5)).mean()*100:.0f}%")

worst = np.argsort(cg)[:24]
show_grid([GEN[i] for i in worst], [f"corr {cg[i]:.2f}" for i in worst],
          ncols=8, size=2.0, save="a5_failures.png")
show_grid(GEN[:48], [str(i) for i in range(48)], ncols=8, size=2.0, save="a5_catalogue.png")

## S_build

Real patches have no silhouette ground truth, so the transfer is checked against the annotated keypoints instead: CEJ and root-tip landmarks should fall on or inside the predicted silhouette.

In [ ]:
dists = collections.defaultdict(list)
ncomp, fills = [], []
for p, m in zip(PATCHES, TOOTH):
    dt = cv2.distanceTransform((1 - m).astype(np.uint8), cv2.DIST_L2, 3)
    for name in ("CEJ_M", "CEJ_D", "RL_M", "RL_D", "RL_C"):
        x, y, v = p["kp"][KP[name]]
        if v > 0 and 0 <= int(y) < PATCH_H and 0 <= int(x) < PATCH_W:
            dists[name].append(float(dt[int(y), int(x)]))
    ncomp.append(cv2.connectedComponents(m)[0] - 1)
    fills.append(m.mean())

print(f"fill fraction: median {np.median(fills):.3f}  p5 {np.percentile(fills, 5):.3f}  "
      f"p95 {np.percentile(fills, 95):.3f}")
print(f"connected components: median {np.median(ncomp):.0f}  frac==1 {np.mean(np.array(ncomp)==1):.2f}")
print("keypoint distance to the silhouette (px, 0 = inside):")
for k, v in dists.items():
    v = np.array(v)
    print(f"  {k:6s} n={len(v):4d}  median {np.median(v):5.1f}  p90 {np.percentile(v, 90):5.1f}"
          f"  frac<=2px {(v <= 2).mean():.2f}")

order = np.argsort([p["pbl"] for p in PATCHES])
sel = [int(order[i]) for i in np.linspace(0, len(order) - 1, 12).astype(int)]
imgs, titles = [], []
for i in sel:
    c = to_canvas(PATCHES[i]["patch"])
    imgs.append(overlay_mask(c, TOOTH[i], ((0, 0, 0), (255, 90, 90))))
    titles.append(f'PBL {PATCHES[i]["pbl"]:.2f}  fill {TOOTH[i].mean():.2f}')
show_grid(imgs, titles, ncols=6, size=2.4, save="b1_s_build.png")

## Masks

In [ ]:
sel = []
for st in range(4):
    sel += [i for i, p in enumerate(PATCHES) if p["stage"] == st and i not in dropped][:3]

imgs, titles = [], []
for i in sel:
    p = PATCHES[i]
    want_i = np.mean([v for v in (p["pbl_m"], p["pbl_d"]) if v is not None])
    c = to_canvas(p["patch"])
    imgs.append(np.hstack([c, overlay_mask(c, MASKS[i])]))
    titles.append(f'{STAGE_NAMES[p["stage"]]}   keypoint {want_i:.2f}   '
                  f'mask {realized_pbl(MASKS[i]):.2f}')
show_grid(imgs, titles, ncols=3, size=3.4, save="b2_masks.png")

# class 2 is thin and is the class every faithfulness number will hinge on
area = np.array([[(MASKS[i] == k).mean() for k in range(N_MASK_CLASSES)] for i in idx])
print("mean area fraction per class:")
for k, n in enumerate(MASK_NAMES):
    print(f"  {k} {n:14s} {area[:, k].mean():.4f}   present in {(area[:, k] > 0).mean()*100:3.0f}% of masks")

## S_eval

The evaluator's own error, setting the floor for every number measured with it.

In [ ]:
PRED_VA = predict_masks(s_eval, EVAL_XV)
D = np.array([dice_per_class(a, b) for a, b in zip(PRED_VA, EVAL_YV)])

print(f"S_eval per-class Dice on VAL (n={len(D)})")
for k, n in enumerate(MASK_NAMES):
    v = D[:, k][~np.isnan(D[:, k])]
    print(f"  {k} {n:14s} mean {v.mean():.4f}  p10 {np.percentile(v, 10):.4f}  n={len(v)}")
print(f"  mean over classes: {np.nanmean(D):.4f}")

got  = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in PRED_VA])
ref  = np.array([realized_pbl(m) for m in EVAL_YV])
want = np.array([np.mean([v for v in (PATCHES[i]["pbl_m"], PATCHES[i]["pbl_d"])
                          if v is not None]) for i in seg_va])
ok = ~np.isnan(got)
print(f"\nrealized_pbl of S_eval on REAL VAL patches, n={ok.sum()}/{len(got)}")
print(f"  vs the constructed mask : MAE {np.abs(got[ok]-ref[ok]).mean():.4f}   "
      f"bias {np.mean(got[ok]-ref[ok]):+.4f}      <- S_eval's own error")
print(f"  vs the keypoint PBL     : MAE {np.abs(got[ok]-want[ok]).mean():.4f}   "
      f"bias {np.mean(got[ok]-want[ok]):+.4f}      <- THE measurement floor")

order = np.argsort([want[i] for i in range(len(seg_va))])
sel = [int(order[i]) for i in np.linspace(0, len(order) - 1, 6).astype(int)]
imgs, titles = [], []
for j in sel:
    c = to_canvas(EVAL_XV[j])
    imgs.append(np.hstack([overlay_mask(c, EVAL_YV[j]), overlay_mask(c, PRED_VA[j])]))
    titles.append(f"built {ref[j]:.2f} | S_eval {got[j]:.2f}   (keypoint {want[j]:.2f})")
show_grid(imgs, titles, ncols=3, size=3.4, save="b3_s_eval.png")

## Conditioned LDM

In [ ]:
M_TE   = MASKS[seg_te]
REAL_C = [PATCHES[i]["patch"] for i in seg_te]

GEN_C = []
for k in range(0, len(seg_te), 25):
    c = cond_channels(torch.from_numpy(M_TE[k:k + 25]).to(DEVICE))
    GEN_C += sample_ldm(unet_c, len(c), steps=100, seed=SEED + k, c=c, w=1.0)
print(f"generated {len(GEN_C)} from TEST masks at w=1.0 (plain conditional, the Konz setting)")

Faithfulness:Dice between `S_eval`'s reading of the generated image and the mask that was commanded, next to the same quantity on real images.

In [ ]:
P_GEN  = predict_masks(s_eval, np.stack(GEN_C))
P_REAL = predict_masks(s_eval, np.stack(REAL_C))

# three Dice columns: what the generated image scores, what the real image scores against
# the same commanded mask (the ceiling: S_eval is imperfect on real data too), and how far
# the generated image is from the real one as S_eval sees them
D_gen   = np.array([dice_per_class(a, b) for a, b in zip(P_GEN,  M_TE)])
D_ceil  = np.array([dice_per_class(a, b) for a, b in zip(P_REAL, M_TE)])
D_pair  = np.array([dice_per_class(a, b) for a, b in zip(P_GEN,  P_REAL)])

print(f"faithfulness on {len(seg_te)} TEST masks     Dice(S_eval(x), commanded mask)")
print(f"{'class':22s} {'generated':>10} {'real = ceiling':>15} {'gen vs real':>12}")
for k, n in enumerate(MASK_NAMES):
    print(f"  {k} {n:18s} {np.nanmean(D_gen[:, k]):10.4f} {np.nanmean(D_ceil[:, k]):15.4f} "
          f"{np.nanmean(D_pair[:, k]):12.4f}")
print(f"  {'mean over classes':20s} {np.nanmean(D_gen):10.4f} {np.nanmean(D_ceil):15.4f} "
      f"{np.nanmean(D_pair):12.4f}")

want  = np.array([realized_pbl(m) for m in M_TE])
got   = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in P_GEN])
ceil_ = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in P_REAL])
ok = ~np.isnan(got) & ~np.isnan(ceil_)

print(f"\nPBL command error, n={ok.sum()}/{len(want)}")
for name, v in (("generated", got), ("real (the floor)", ceil_)):
    e = np.abs(v[ok] - want[ok])
    print(f"  {name:18s} MAE {e.mean():.4f}  median {np.median(e):.4f}  "
          f"p90 {np.percentile(e, 90):.4f}  bias {np.mean(v[ok] - want[ok]):+.4f}  "
          f"corr {np.corrcoef(v[ok], want[ok])[0, 1]:+.3f}")

cm = np.zeros((4, 4), int)
for a, b in zip(want[ok], got[ok]):
    cm[stage_of(a), stage_of(b)] += 1
print(f"\nstage confusion, commanded (rows) vs realized (cols)   agreement "
      f"{np.trace(cm)/cm.sum():.3f}")
print(f"  {'':10s}" + "".join(f"{n[:4]:>7}" for n in STAGE_NAMES))
for i, n in enumerate(STAGE_NAMES):
    print(f"  {n:10s}" + "".join(f"{cm[i, j]:7d}" for j in range(4)))

FID, KID and diversity on the same 126 test masks.

In [ ]:
fid_c, kid_c, kid_c_sd = fid_kid(REAL_C, GEN_C)
print(f"conditioned LDM  FID {fid_c:6.1f} | KID {kid_c:.4f} +- {kid_c_sd:.4f}   (n={len(REAL_C)})")
print(f"unconditional    FID {fid_g:6.1f} | KID {kid_g:.4f} +- {kid_sd:.4f}")
print(f"decoder only     FID {rfid:6.1f} | KID {rkid:.4f} +- {rkid_sd:.4f}")

sub = list(range(0, len(seg_te), max(1, len(seg_te) // 24)))[:24]
div = []
for j in sub:
    c = cond_channels(torch.from_numpy(M_TE[j:j + 1]).to(DEVICE)).repeat(8, 1, 1, 1)
    div.append(lpips_diversity(sample_ldm(unet_c, 8, steps=100, seed=SEED + 7919 * j, c=c),
                               n_pairs=28))
print(f"\nLPIPS diversity, 8 samples per mask over {len(sub)} masks: {np.mean(div):.3f} "
      f"+-{np.std(div):.3f}   (unconditional {div_g:.3f}, real TEST {div_r:.3f})")

o = np.argsort(want)
sel = [int(o[i]) for i in np.linspace(0, len(o) - 1, 5).astype(int)]
imgs, titles = [], []
for j in sel:
    c = cond_channels(torch.from_numpy(M_TE[j:j + 1]).to(DEVICE)).repeat(3, 1, 1, 1)
    s3 = sample_ldm(unet_c, 3, steps=100, seed=SEED + j, c=c)
    row = [overlay_mask(to_canvas(REAL_C[j]), M_TE[j]), to_canvas(REAL_C[j])] + [to_canvas(a) for a in s3]
    imgs.append(np.hstack(row))
    titles.append(f"commanded {want[j]:.2f} -> realized {got[j]:.2f}   "
                  f"mask | real | 3 samples")
show_grid(imgs, titles, ncols=1, size=9.0, save="b5_faithfulness.png")

Guidance: the faithfulness vs. diversity trade-off does not actually exist here since
`w=1` is simultaneously the most faithful, the less diverse and the one with the smallest FID.


In [ ]:
W_GRID = (0.0, 1.0, 2.0, 3.0, 5.0, 8.0)
div_masks = list(range(0, len(seg_te), max(1, len(seg_te) // 8)))[:8]
SWEEP = {}

for w in W_GRID:
    gen = []
    for k in range(0, len(seg_te), 25):
        c = cond_channels(torch.from_numpy(M_TE[k:k + 25]).to(DEVICE))
        gen += sample_ldm(unet_c, len(c), steps=100, seed=SEED + k, c=c, w=w)
    pg = predict_masks(s_eval, np.stack(gen))
    d2 = np.nanmean([dice_per_class(a, b)[2] for a, b in zip(pg, M_TE)])
    got_w = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in pg])
    ok_w = ~np.isnan(got_w)
    dv = []
    for j in div_masks:   # 8 samples from one mask, at this w
        c8 = cond_channels(torch.from_numpy(M_TE[j:j + 1]).to(DEVICE)).repeat(8, 1, 1, 1)
        dv.append(lpips_diversity(sample_ldm(unet_c, 8, steps=100, seed=SEED + 7919 * j,
                                             c=c8, w=w), n_pairs=28))
    SWEEP[w] = dict(dice2=float(d2),
                    mae=float(np.abs(got_w[ok_w] - want[ok_w]).mean()),
                    corr=float(np.corrcoef(got_w[ok_w], want[ok_w])[0, 1]),
                    bias=float(np.mean(got_w[ok_w] - want[ok_w])),
                    agr=float(np.mean([stage_of(a) == stage_of(b) for a, b in zip(want[ok_w], got_w[ok_w])])),
                    div=float(np.mean(dv)), fid=float(fid_kid(REAL_C, gen)[0]))
    r = SWEEP[w]
    print(f"w={w:<4} dice2 {r['dice2']:.4f}  PBL MAE {r['mae']:.4f}  corr {r['corr']:+.3f}  "
          f"bias {r['bias']:+.4f}  stage {r['agr']:.3f}  div {r['div']:.3f}  FID {r['fid']:.1f}")

print(f"\nreference: real images through the same S_eval -> MAE 0.0380  corr +0.876  stage 0.810")

In [ ]:
ws = list(W_GRID)
div_, corr_, fid_ = ([SWEEP[w][k] for w in ws] for k in ("div", "corr", "fid"))
fig, ax = plt.subplots(1, 3, figsize=(16, 3.9))

ax[0].plot(ws, corr_, "o-", color="crimson", label="PBL corr")
ax[0].plot(ws, [SWEEP[w]["agr"] for w in ws], "s-", color="steelblue", label="stage agreement")
ax[0].axhline(0.876, color="crimson", ls=":", lw=1)  # what real images score
ax[0].axhline(0.810, color="steelblue", ls=":", lw=1)
ax[0].axhline(corr_[0], color="k", ls="--", lw=1) # w=0: what "mask ignored" looks like
ax[0].set_xlabel("guidance w"); ax[0].legend(fontsize=7)
ax[0].set_title("faithfulness (dotted = real images, dashed = mask ignored)", fontsize=9)

# diversity alone is misleading past w=1: it rises because the images fall apart, not
# because they vary. FID on the twin axis is what makes that readable.
ax[1].plot(ws, div_, "o-", color="seagreen", label="LPIPS diversity")
ax[1].axhline(div_r, color="seagreen", ls=":", lw=1) # real test
ax[1].set_xlabel("guidance w"); ax[1].set_ylabel("LPIPS diversity", color="seagreen")
tw = ax[1].twinx()
tw.plot(ws, fid_, "^--", color="darkorange", label="FID")
tw.set_ylabel("FID", color="darkorange")
ax[1].set_title("diversity rises past w=1 only as FID does", fontsize=9)

sc = ax[2].scatter(div_, corr_, c=fid_, cmap="viridis_r", s=110, zorder=3)
ax[2].plot(div_, corr_, "-", color="grey", lw=0.8, alpha=0.6, zorder=2)
for w in ws:
    ax[2].annotate(f"w={w:g}", (SWEEP[w]["div"], SWEEP[w]["corr"]), fontsize=7,
                   textcoords="offset points", xytext=(6, 5))
ax[2].axhline(corr_[0], color="k", ls="--", lw=1)
ax[2].set_xlabel("LPIPS diversity"); ax[2].set_ylabel("PBL correlation")
ax[2].set_title("no trade-off: w=1 dominates, the rest is degradation", fontsize=9)
fig.colorbar(sc, ax=ax[2], label="FID (lower is better)")

for a in (ax[0], ax[1], ax[2]):
    a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGS / "b6_guidance.png", dpi=130, bbox_inches="tight")

best_w = max(W_GRID, key=lambda w: SWEEP[w]["corr"])
print(f"operating point w={best_w:g}: corr {SWEEP[best_w]['corr']:+.3f}, "
      f"MAE {SWEEP[best_w]['mae']:.4f}, diversity {SWEEP[best_w]['div']:.3f}, "
      f"FID {SWEEP[best_w]['fid']:.1f}")
print("w=1 is simultaneously the most faithful, the least diverse and the lowest FID:")
print("  the curve the assignment asks for exists and is degenerate, which is the result.")

## Latent vs. pixel space

In [ ]:
GEN_PX = []
for k in range(0, len(seg_te), 12):
    c = cond_channels(torch.from_numpy(M_TE[k:k + 12]).to(DEVICE), hw=(PATCH_H, PATCH_W))
    GEN_PX += sample_ldm(unet_px, len(c), steps=100, seed=SEED + k, c=c, w=1.0, latent=False)

P_PX = predict_masks(s_eval, np.stack(GEN_PX))
D_px = np.array([dice_per_class(a, b) for a, b in zip(P_PX, M_TE)])
got_px = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in P_PX])
ok_px = ~np.isnan(got_px)
fid_px, kid_px, kid_px_sd = fid_kid(REAL_C, GEN_PX)

print(f"H1: latent LDM vs pixel DDPM, {len(seg_te)} TEST masks, w=1.0")
print(f"{'':26s} {'LDM (latent)':>14} {'DDPM (pixel)':>14} {'real = ceiling':>15}")
print(f"{'Dice, exposed root':26s} {np.nanmean(D_gen[:, 2]):14.4f} {np.nanmean(D_px[:, 2]):14.4f} "
      f"{np.nanmean(D_ceil[:, 2]):15.4f}")
print(f"{'Dice, mean over classes':26s} {np.nanmean(D_gen):14.4f} {np.nanmean(D_px):14.4f} "
      f"{np.nanmean(D_ceil):15.4f}")
print(f"{'PBL MAE':26s} {np.abs(got[ok]-want[ok]).mean():14.4f} "
      f"{np.abs(got_px[ok_px]-want[ok_px]).mean():14.4f} 0.0380".rjust(0))
print(f"{'PBL correlation':26s} {np.corrcoef(got[ok],want[ok])[0,1]:+14.3f} "
      f"{np.corrcoef(got_px[ok_px],want[ok_px])[0,1]:+14.3f} {0.876:+15.3f}")
print(f"{'PBL bias':26s} {np.mean(got[ok]-want[ok]):+14.4f} "
      f"{np.mean(got_px[ok_px]-want[ok_px]):+14.4f} {0.0004:+15.4f}")
print(f"{'stage agreement':26s} "
      f"{np.mean([stage_of(a)==stage_of(b) for a,b in zip(want[ok],got[ok])]):14.3f} "
      f"{np.mean([stage_of(a)==stage_of(b) for a,b in zip(want[ok_px],got_px[ok_px])]):14.3f} "
      f"{0.810:15.3f}")
print(f"{'FID':26s} {fid_c:14.1f} {fid_px:14.1f}")
print(f"{'KID':26s} {kid_c:14.4f} {kid_px:14.4f}")
print(f"\nboth trained 10000 optimizer steps at effective batch 32, same scheduler, "
      f"same MAT, same DDIM 100")

o = np.argsort(want)
sel = [int(o[i]) for i in np.linspace(0, len(o) - 1, 5).astype(int)]
imgs, titles = [], []
for j in sel:
    imgs.append(np.hstack([overlay_mask(to_canvas(REAL_C[j]), M_TE[j]), to_canvas(REAL_C[j]),
                           to_canvas(GEN_C[j]), to_canvas(GEN_PX[j])]))
    titles.append(f"commanded {want[j]:.2f}   mask | real | LDM {got[j]:.2f} | "
                  f"pixel DDPM {got_px[j]:.2f}")
show_grid(imgs, titles, ncols=1, size=8.0, save="c1_h1.png")

In [ ]:
rec = []
for i in range(0, len(REAL_C), 16):
    x = torch.stack([torch.from_numpy(norm_img(a))[None] for a in REAL_C[i:i + 16]]).to(DEVICE)
    with torch.no_grad(), torch.autocast(DEVICE, dtype=DTYPE):
        rec += [denorm_img(a) for a in vae(x).sample.float()[:, 0].cpu().numpy()]

P_REC   = predict_masks(s_eval, np.stack(rec))
D_rec   = np.array([dice_per_class(a, b) for a, b in zip(P_REC, M_TE)])
got_rec = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in P_REC])
ok_rec  = ~np.isnan(got_rec)

print("where the latent pipeline loses faithfulness")
print(f"{'':34s} {'Dice cl.2':>10} {'PBL MAE':>9} {'bias':>9} {'corr':>8}")
for name, D, v, k in (("real patches",              D_ceil, ceil_,  ok),
                      ("+ VAE round-trip only",     D_rec,  got_rec, ok_rec),
                      ("+ diffusion in the latent", D_gen,  got,    ok),
                      ("diffusion in pixels",       D_px,   got_px, ok_px)):
    print(f"  {name:32s} {np.nanmean(D[:, 2]):10.4f} {np.abs(v[k]-want[k]).mean():9.4f} "
          f"{np.mean(v[k]-want[k]):+9.4f} {np.corrcoef(v[k], want[k])[0, 1]:+8.3f}")

print(f"\n{'class':22s} {'LDM':>8} {'DDPM px':>9} {'real':>8}   (Konz pixel, 12000 img: 0.898-0.903)")
for k, n in enumerate(MASK_NAMES):
    print(f"  {k} {n:18s} {np.nanmean(D_gen[:, k]):8.4f} {np.nanmean(D_px[:, k]):9.4f} "
          f"{np.nanmean(D_ceil[:, k]):8.4f}")
print(f"  {'mean':20s} {np.nanmean(D_gen):8.4f} {np.nanmean(D_px):9.4f} {np.nanmean(D_ceil):8.4f}")

## Retargeting the masks

the crest commanded apically on 16 teeth, five targets spanning 0.10–0.85, on masks whose severity the model has essentially never seen. The retargeted masks are checked first: if they did not carry the commanded PBL, a failure downstream would say nothing about the generator.

In [ ]:
# One tooth, the crest commanded down.Same silhouette, same seed, only pbl_target changes,
# so any difference in the output is attributable to the command and to nothing else.
TARGETS = (0.10, 0.30, 0.50, 0.70, 0.85)
ct_src = [seg_tr[j] for j in np.linspace(0, len(seg_tr) - 1, 16).astype(int)]
ct_masks = np.stack([retarget_mask(MASKS[i], t) for t in TARGETS for i in ct_src])

# read PBL back out of the retargeted masks and compare with what was commanded.
ct_want = np.repeat(TARGETS, len(ct_src))
ct_mpbl = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in ct_masks])
print(f"retarget_mask: commanded vs mask PBL  MAE "
      f"{np.nanmean(np.abs(ct_mpbl - ct_want)):.4f}  n={int((~np.isnan(ct_mpbl)).sum())}/{len(ct_masks)}")

ct = {}
for name, net, lat, b in (("latent", unet_c, True, 25), ("pixel", unet_px, False, 12)):
    imgs = gen_bank(net, ct_masks, lat, True, b, seed=SEED + 101)
    got_ct = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan
                       for m in predict_masks(s_eval, imgs)])
    ct[name] = (imgs, got_ct)
    w_ct = np.repeat(TARGETS, len(ct_src))
    k = ~np.isnan(got_ct)
    print(f"{name:7s} commanded -> realized: corr {np.corrcoef(got_ct[k], w_ct[k])[0,1]:+.3f}  "
          f"MAE {np.abs(got_ct[k]-w_ct[k]).mean():.4f}")
    print("   " + "  ".join(f"{t:.2f}->{np.nanmean(got_ct[i*len(ct_src):(i+1)*len(ct_src)]):.3f}"
                            for i, t in enumerate(TARGETS)))

j = 3
imgs, titles = [], []
for i, t in enumerate(TARGETS):
    k = i * len(ct_src) + j
    imgs.append(np.hstack([overlay_mask(to_canvas(PATCHES[ct_src[j]]["patch"]), ct_masks[k]),
                           to_canvas(ct["latent"][0][k]), to_canvas(ct["pixel"][0][k])]))
    titles.append(f"commanded PBL {t:.2f}   mask | latent {ct['latent'][1][k]:.2f} | "
                  f"pixel {ct['pixel'][1][k]:.2f}")
show_grid(imgs, titles, ncols=1, size=6.5, save="c2_retarget.png")

In [ ]:
# The pooled correlation mixes two things: how much one tooth responds to the command, and
# how much teeth differ from each other. The same five targets are commanded on every tooth,
# so each carries its own offset and the pooled number is dragged down by it. What decides
# whether the command works is the WITHIN-tooth slope.
def within_tooth(got, targets=TARGETS, n_src=None):
    """got: realized PBL in the order [target, tooth]. Returns pooled r, mean within-tooth
       r, mean slope (1.0 = perfect obedience), monotone count, teeth used."""
    G = got.reshape(len(targets), n_src or len(got) // len(targets))
    t = np.array(targets)
    rs, sl, mono = [], [], 0
    for c in range(G.shape[1]):
        v = G[:, c]
        if np.isnan(v).any():
            continue
        rs.append(np.corrcoef(t, v)[0, 1])
        sl.append(np.polyfit(t, v, 1)[0])
        mono += bool(np.all(np.diff(v) >= -0.02))
    f = ~np.isnan(G.ravel())
    pooled = np.corrcoef(np.repeat(t, G.shape[1])[f], G.ravel()[f])[0, 1]
    return pooled, float(np.mean(rs)), float(np.mean(sl)), mono, G.shape[1]

print(f"{'':8s} {'pooled r':>9} {'within-tooth r':>15} {'slope':>8} {'monotone':>10}")
for name in ("latent", "pixel"):
    p, r, s, mo, n = within_tooth(ct[name][1], n_src=len(ct_src))
    print(f"  {name:6s} {p:+9.3f} {r:+15.3f} {s:8.3f} {mo:6d}/{n:<3d}")


In [ ]:
# The banks carry the commanded severity, but a classifier sees the image.
print(f"\n{'bank':8s} {'realized PBL':>14} {'>= 0.33':>9} {'>= 0.66':>9}   "
      f"(commanded: 100% >= 0.33, 60% >= 0.66)")
for k, p in BANK_PBL.items():
    ok_b = ~np.isnan(p)
    print(f"  {k:6s} {np.nanmean(p):6.3f} +-{np.nanstd(p):5.3f} "
          f"{(p[ok_b] >= ADVANCED_MIN).mean()*100:8.0f}% {(p[ok_b] >= SEVERE_MIN).mean()*100:8.0f}%"
          f"   {(~ok_b).sum()} unreadable")
print(f"  {'real TR':6s} {np.mean([np.mean([x for x in (PATCHES[i]['pbl_m'], PATCHES[i]['pbl_d']) if x is not None]) for i in seg_tr]):6.3f}"
      f"          {np.mean([PATCHES[i]['pbl'] >= ADVANCED_MIN for i in seg_tr])*100:8.0f}%"
      f" {np.mean([PATCHES[i]['pbl'] >= SEVERE_MIN for i in seg_tr])*100:8.0f}%")


## Downstream task

Binary classification between normal and advanced on tooth patches, mAP on a strict holdout of 126 patches. 4 different banks are used.

In [ ]:
print(f"{'arm':34s} {'n syn':>6} {'lab Adv':>8} {'true Adv':>9} "
      f"{'mAP':>14} {'macro-F1':>14} {'AP Advanced':>14}")
for name, rows in CLF.items():
    m = {k: np.array([r[k] for r in rows]) for k in ("mAP", "f1", "ap_adv")}
    n, lab, true = ARM_INFO[name]
    print(f"  {name:32s} {n:6d} {'--' if np.isnan(lab) else f'{lab*100:.0f}%':>8} "
          f"{'--' if np.isnan(true) else f'{true*100:.0f}%':>9} "
          f"{m['mAP'].mean():6.3f}+-{m['mAP'].std():.3f} "
          f"{m['f1'].mean():6.3f}+-{m['f1'].std():.3f} "
          f"{m['ap_adv'].mean():6.3f}+-{m['ap_adv'].std():.3f}")
print(f"\nbaseline: all-Advanced predictor scores AP {test_y.mean():.3f}; "
      f"{test_y.sum()}/{len(test_y)} of TEST is Advanced; real TRAIN has "
      f"{real_y.sum()} Advanced in {len(real_y)}")

# Seeds are paired (same init, same batch order), so the per-seed difference is a much
# tighter instrument than the bars: it cancels the seed effect the two arms share.
names = list(CLF.keys())
def paired(a, b, key):
    return np.array([x[key] - y[key] for x, y in zip(CLF[a], CLF[b])])

def find(tag):
    return next(n for n in names if n.startswith(tag))

CONTRASTS = [(n, names[0], "vs real only") for n in names[1:]] + [
    (find("B_px"),  find("D_px"),  "guidance, intent labels"),
    (find("B_lat"), find("D_lat"), "guidance, intent labels"),
    (find("E_px"),  find("F_px"),  "guidance, measured labels"),
    (find("E_lat"), find("F_lat"), "guidance, measured labels"),
    (find("E_px"),  find("B_px"),  "cost of the label noise"),
    (find("E_lat"), find("B_lat"), "cost of the label noise"),
    (find("G_px"),  find("B_px"),  "filtered vs unfiltered"),
    (find("G_lat"), find("B_lat"), "filtered vs unfiltered")]

print(f"\n{'contrast':22s} {'d mAP':>16} {'d AP Advanced':>16} {'wins':>6}   what it tests")
for a, b, what in CONTRASTS:
    dm, da = paired(a, b, "mAP"), paired(a, b, "ap_adv")
    print(f"  {a.split()[0]:>6s} - {b.split()[0]:<12s} {dm.mean():+6.3f}+-{dm.std():.3f} "
          f"{da.mean():+6.3f}+-{da.std():.3f} {(dm > 0).sum():4d}/5   {what}")

key_ix = [i for i, c in enumerate(CONTRASTS) if c[2] != "vs real only"]
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for row, ix in enumerate((list(range(len(names) - 1)), key_ix)):
    for a, key, ttl in zip(ax[row], ("mAP", "ap_adv"), ("mAP", "AP on Advanced")):
        v = [paired(CONTRASTS[i][0], CONTRASTS[i][1], key) for i in ix]
        lab = [f"{CONTRASTS[i][0].split()[0]}-{CONTRASTS[i][1].split()[0]}" for i in ix]
        a.bar(range(len(v)), [x.mean() for x in v], yerr=[x.std() for x in v],
              capsize=4, color="grey" if row == 0 else "firebrick")
        for i, x in enumerate(v):
            a.scatter([i] * len(x), x, s=9, color="k", zorder=3)
        a.axhline(0, color="k", lw=1)
        a.set_xticks(range(len(v))); a.set_xticklabels(lab, fontsize=7, rotation=30, ha="right")
        sub = "every arm vs real only" if row == 0 else "the controlled contrasts"
        a.set_title(f"paired per-seed delta, {ttl}  --  {sub}", fontsize=9)
        a.grid(alpha=0.3, axis="y")
fig.tight_layout(); fig.savefig(FIGS / "c3_downstream.png", dpi=130, bbox_inches="tight")


## Anatomical loss

In [ ]:
# Does an explicit constraint recover the spatial precision the latent space loses, and does
# it do the same for the pixel model?
GEN_A  = np.stack(gen_bank(unet_a,    M_TE, True,  True, 25))
CT_A   = np.stack(gen_bank(unet_a,    ct_masks, True,  True, 25, seed=SEED + 101))
GEN_PA = np.stack(gen_bank(unet_px_a, M_TE, False, True, 12))
CT_PA  = np.stack(gen_bank(unet_px_a, ct_masks, False, True, 12, seed=SEED + 101))

MODELS = (("latent, plain",      GEN_C,  np.stack(ct["latent"][0])),
          ("latent + anatomic",  GEN_A,  CT_A),
          ("pixel, plain",       np.stack(GEN_PX), np.stack(ct["pixel"][0])),
          ("pixel + anatomic",   GEN_PA, CT_PA),
          ("real = ceiling",     np.stack(REAL_C), None))

for seg, seg_name in ((s_eval, "S_eval  (in the loss)"), (s_eval2, "S_eval2 (independent)")):
    print(f"\n{seg_name}     {'Dice exposed root':>18} {'Dice mean':>10} "
          f"{'PBL MAE':>9} {'PBL bias':>9} {'corr':>7}")
    for name, imgs, _ in MODELS:
        p = predict_masks(seg, imgs)
        d = np.array([dice_per_class(a, b) for a, b in zip(p, M_TE)])
        g = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan for m in p])
        k = ~np.isnan(g)
        print(f"  {name:26s} {np.nanmean(d[:, 2]):18.4f} {np.nanmean(d):10.4f} "
              f"{np.abs(g[k] - want[k]).mean():9.4f} {np.mean(g[k] - want[k]):+9.4f} "
              f"{np.corrcoef(g[k], want[k])[0, 1]:+7.3f}")

print(f"\n{'model':20s} {'judge':>9} {'pooled r':>9} {'within-tooth r':>15} "
      f"{'slope':>8} {'monotone':>10}")
for name, _, imgs in MODELS:
    if imgs is None:
        continue
    for seg, seg_name in ((s_eval, "S_eval"), (s_eval2, "S_eval2")):
        got = np.array([realized_pbl(m) if realized_pbl(m) is not None else np.nan
                        for m in predict_masks(seg, imgs)])
        p, r, s, mo, n = within_tooth(got, n_src=len(ct_src))
        print(f"  {name:18s} {seg_name:>9} {p:+9.3f} {r:+15.3f} {s:8.3f} {mo:6d}/{n:<3d}")

print("\nHF energy at sigma=1.5   " + "   ".join(
    f"{n} {np.mean([hf_energy(a) for a in v]):.3f}" for n, v, _ in MODELS))

print(f"\n{'model':22s} {'FID':>8} {'KID':>18}")
for name, imgs, _ in MODELS[:-1]:
    f, k, ksd = fid_kid(REAL_C, list(imgs))
    print(f"  {name:20s} {f:8.1f} {k:10.4f} +-{ksd:.4f}")

j = 3
imgs, titles = [], []
for i, t in enumerate(TARGETS):
    k = i * len(ct_src) + j
    imgs.append(np.hstack([overlay_mask(to_canvas(PATCHES[ct_src[j]]["patch"]), ct_masks[k])]
                          + [to_canvas(v[k]) for _, _, v in MODELS if v is not None]))
    titles.append(f"commanded PBL {t:.2f}   mask | lat | lat+anat | px | px+anat")
show_grid(imgs, titles, ncols=1, size=8.0, save="c4_anatomical.png")
